# Flight Delay Predictions - Team 1_1 - The Lazy Evaluators

## Team Members

| Name           | Email                |
|----------------|----------------------|
| Archis Dhar         | archis_dhar@berkeley.edu     |
| Andrew Saxe         | andrewsaxe@berkeley.edu      |
| Nithya Nalluri      |  nithya_nalluri@berkeley.edu |
| Mario Blanco-Alcala | marioba@berkeley.edu         |
| Sam Cook            | sacook828@berkeley.edu       |

<img src="https://raw.githubusercontent.com/sacook828-berkeley/261_final_proj_images/main/all_team_pics.png">

### Phase Leader Plan  

| Week / Dates | Task  | Primary  | Secondary Support |
|------|-------------|-------------|---------|
| Week 0: 3/08-3/15  | Phase I | Archis Dhar | Andrew Saxe |
| Week 1: 3/15-3/22  | Phase II | Nithya Nalluri | Mario Blanco-Alcala |
| Week 2: 3/22-3/29  | Phase II | Nithya Nalluri | Sam Cook |
| Week 3: 3/29-4/05  | Homework 5  | N/A | N/A |
| Week 4: 4/05-4/12  | Phase III | Mario Blanco-Alcala | Archis Dhar |
| Week 5: 4/12-4/19  | Phase III  | Sam Cook | Nithya Nalluri |


### Credit Assignment Plan

| Team Member | Phase 3 Work Breakdown | Total Hours |
| :--- | :--- | :--- |
| **Archis Dhar** | • **MLP Neural Network Development (Stage 1):** Designed and implemented all MLP classification architectures (MLP-A, MLP-B, MLP-C) using PySpark MLlib MultilayerPerceptronClassifier with warm-start early stopping, expanding-window temporal CV, and 1:1 undersampling. Built the MLP-D comparison experiment using only 60 dense numeric features (18 hrs)<br>• **MLP Neural Network Development (Stage 2):** Built PyTorch regression MLPs (MLP-A Shallow, MLP-B Deeper) with Adam optimizer, batch normalization, dropout, ReduceLROnPlateau scheduler, and Clip@P99 target transformation on driver-node CPU (15 hrs)<br>• **Feature Engineering:** Developed weather composite features including origin and destination severity indices, weather differential, precipitation flags, and low-visibility indicators (5 hrs)<br>• **Presentation:** Designed and built the majority of Phase 3 presentation slides (4 hrs)<br>• **Technical Writing:** Authored ML Pipeline, Modeling Decisions, Results and Findings, Conclusion, and Next Steps sections of the Phase 3 report (8 hrs)<br>• **Report Review:** Reviewed and edited the final Phase 3 report for clarity and consistency (4 hrs) | **54 hrs** |
| **Nithya Nalluri** | • **GBT Pipeline Development:** Built and optimized the GBT classification and regression pipelines with extended hyperparameter grids, threshold tuning, and log-transform variant for Stage 2 (13 hrs)<br>• **Pipeline Engineering:** Designed the two-stage connected pipeline architecture with out-of-fold data preparation, checkpoint strategy, and temporal cross-validation framework (15 hrs)<br>• **Data Preprocessing:** Implemented three-stage weather imputation (forward-fill, airport-month mean, global median) and categorical encoding pipeline with StringIndexer, OneHotEncoder, VectorAssembler, and StandardScaler (10 hrs)<br>• **Presentation:** Contributed to Phase 3 presentation slides (3 hrs)<br>• **Technical Writing & Diagrams:** Reviewed and edited the final Phase 3 report for clarity and consistency. Created the pipeline architecture diagram illustrating the full two-stage data flow from ingestion through classification and regression (6 hrs)<br>• **Report Review:** Performed final review and editing of the Phase 3 report for clarity and consistency (3 hrs) | **50 hrs** |
| **Mario Blanco-Alcala** | • **Exploratory Data Analysis:** Performed EDA on the 60-month OTPW dataset, analyzing feature distributions, delay rate patterns across years, class imbalance characteristics, and weather variable correlations to inform modeling decisions (8 hrs)<br>• **Technical Writing (EDA):** Authored the Exploratory Data Analysis section of the Phase 3 report documenting findings and visualizations (4 hrs)<br>• **Presentation:** Contributed to Phase 3 presentation slides (2 hrs)<br>• **Report Review:** Reviewed and edited the final Phase 3 report for clarity and consistency (2 hrs)<br><br>*Personal circumstances limited my ability to help as much this phase* | **16 hrs** |
| **Sam Cook** | • **Graph Feature Engineering:** Implemented PageRank on the directed flight network, computing origin and destination PageRank, total degree, route PageRank difference, and carrier-specific PageRank using training data only (4 hrs)<br>• **GBT Pipeline Experimentation:** Reworked and retrained the GBT classification model with new Graph and other engineered event features (4 hrs)<br>• **Technical Writing (Data Description):** Drafted the formal BTS and NOAA data description sections covering dataset provenance, column definitions, and join methodology (4 hrs)<br>• **Presentation:** Contributed to Phase 3 presentation slides (4 hrs)<br>• **Report Review:** Reviewed and edited the final Phase 3 report for clarity and consistency (4 hrs) | **20 hrs** |
| **Andrew Saxe** | • **Report Review:** Reviewed and edited the final Phase 3 report for clarity and consistency (2 hrs)<br>• **EDA Support:** Assisted with exploratory analysis validating feature distributions on the 60-month dataset (1 hr) | **3 hrs** |

## Abstract

Flight delays do not just inconvenience passengers — they cascade across entire airline networks, creating operational chaos that costs the U.S. aviation industry billions of dollars each year. When one flight falls behind, the ripple effects hit gate assignments, crew schedules, and connecting passengers at airports downstream. Our project tackles this problem from two angles using 31.2 million U.S. domestic flight records (2015–2019) from the Bureau of Transportation Statistics (BTS) joined with hourly weather observations from the National Oceanic and Atmospheric Administration (NOAA): first, predicting whether a flight will be delayed, and second, estimating how long that delay will last. The two key stakeholders are airline operations teams, who need advance warning to reassign gates, reposition crews, and manage network flow, and passengers, who need reliable delay estimates at check-in to decide whether to rebook or wait it out.

For airline operations, our Stage 1 classifier predicts whether a flight will depart at least 15 minutes late, issuing that prediction 2 hours before departure so teams have time to act. We select AUC-PR as the primary decision metric because it evaluates how effectively the model identifies the minority delayed class without inflation from the 82% on-time majority. Our best model achieves an AUC-PR of 0.557, which represents meaningful predictive power given that a random classifier would score only around 0.18 (the base delay rate). For passengers, our Stage 2 regressor estimates how many minutes a flagged flight will actually be delayed. We select MAPE as the primary regression metric because it measures prediction error as a proportion of the actual delay, aligning with how operations teams allocate resources proportionally to expected delay severity. Our best model achieves a MAPE of 61.62%, meaning the predicted delay magnitude differs from the actual delay magnitude by roughly 60% on average. Both metrics show meaningful improvement over our Phase 2 baseline models: Stage 1 AUC-PR improved by 5.8 percentage points and Stage 2 MAPE improved by 31 percentage points.

The remainder of this report walks through how we arrived at these results. We begin with the data description and exploratory analysis, then detail the full machine learning pipeline including preprocessing, feature engineering, and our two-stage modeling approach. We present results and findings for both classification and regression, discuss what worked and what the models still struggle with, and close with next steps for future improvement.

### Glossary of Key Terms

| Abbreviation | Full Name | Description |
| --- | --- | --- |
| **BTS** | Bureau of Transportation Statistics | U.S. government source for the flight data |
| **NOAA** | National Oceanic and Atmospheric Administration | U.S. government source for the weather data |
| **EDA** | Exploratory Data Analysis | Initial investigation of data to discover patterns, spot anomalies, and check assumptions |
| **MLP** | Multilayer Perceptron | A neural network with one or more hidden layers of neurons |
| **GBT** | Gradient Boosted Trees | An ensemble model that builds many decision trees one after another, each correcting the mistakes of the last |
| **P@99** | 99th Percentile | The value below which 99% of the data falls; used here to clip extreme outlier delays |
| **OHE** | One-Hot Encoding | Converting categories (e.g., airport codes) into binary 0/1 columns so models can use them |
| **AUC-ROC** | Area Under the ROC Curve | Measures how well the model separates delayed vs. on-time flights across all decision thresholds |
| **AUC-PR** | Area Under the Precision-Recall Curve | Like AUC-ROC but better suited for imbalanced classes (few delays vs. many on-time) |
| **MAPE** | Mean Absolute Percentage Error | Average prediction error expressed as a percentage of the true value |
| **RMSE** | Root Mean Square Error | Average prediction error in the same units as the target, penalizing large errors more heavily |
| **MAE** | Mean Absolute Error | Average absolute difference between predicted and actual values |
| **CV** | Cross-Validation | Splitting training data into multiple folds to evaluate the model without touching the test set |
| **L-BFGS** | Limited-memory BFGS | An optimization algorithm used to train MLPs efficiently in Spark ML |
| **LR** | Learning Rate | How big a step the model takes when updating its weights each training iteration |
| **PageRank** | PageRank | A graph algorithm (originally from Google) that ranks airport importance based on flight connections |

# Data Description and EDA

   
## 1. Data Description and Source

The flight data for this project originates from the Bureau of Transportation Statistics (BTS) under the U.S. Department of Transportation and contains every domestic US flight with scheduled and actual departure/arrival times, delay causes, carrier info, and route details. The weather data originates from the National Oceanic and Atmospheric Administration (NOAA) repository and contains hourly and daily weather observations from weather stations closest to each airport. This report focuses on the full 5-year pre-joined flight and weather dataset spanning 2015–2019, which integrates flight-level records from the BTS dataset with hourly weather observations from nearby stations. This enables us to incorporate meteorological features alongside operational flight attributes. The integrated flight and weather datasets are provided as Apache Parquet files hosted on the Databricks platform—a cloud-based, unified analytics environment designed for large-scale data processing. The raw constituent datasets—Flights, Weather, and Stations—are available separately as Apache Parquet files. However, these will not be used directly for modeling, as they served as the raw components that were joined and cleaned to create the primary dataset for this work.

---
#### Dataset
This is the primary dataset we built our models on. It covers every domestic flight from 2015 all the way through 2019. By using this full 5-year window, we can make sure our training and testing are solid enough to handle real-world flight patterns.

**Summary of the 5-Year Master Modeling Dataset (2015–2019)**
| Dataset | DBFS Path | Rows | Columns | Format |
| :--- | :--- | :--- | :--- | :--- |
| 60M Joined Flight/Weather (2015–2019) | `dbfs:/mnt/mids-w261/OTPW_60M_Backup/` | 31,673,119 | 214 | Parquet |

---
#### Raw Dataset Column Families
The 214 raw columns in our dataset fall into distinct families based on the type of information they provide. This categorization helps us understand the data landscape and identify which columns are safe to use as model inputs versus those that introduce data leakage.

**Raw Dataset Column Categorization (214 Columns)**
| Family | Example Columns | Description |
| --- | --- | --- |
| Temporal | `YEAR`, `QUARTER`, `MONTH`, `DAY_OF_MONTH`, `DAY_OF_WEEK`, `FL_DATE` | Calendar identifiers for seasonality analysis |
| Carrier / Flight ID | `OP_CARRIER`, `OP_UNIQUE_CARRIER`, `TAIL_NUM`, `OP_CARRIER_FL_NUM` | Airline and aircraft identifiers |
| Route / Airport | `ORIGIN`, `DEST`, `ORIGIN_CITY_NAME`, `DEST_CITY_NAME`, `DISTANCE`, `DISTANCE_GROUP` | Origin-destination pair and route distance |
| Schedule | `CRS_DEP_TIME`, `CRS_ARR_TIME`, `CRS_ELAPSED_TIME` | Planned departure, arrival, block time |
| Actual Performance | `DEP_TIME`, `ARR_TIME`, `DEP_DELAY`, `ARR_DELAY`, `ACTUAL_ELAPSED_TIME`, `AIR_TIME` | Observed performance (post-hoc / leakage) |
| Delay Targets | `DEP_DEL15` (binary), `DEP_DELAY` (continuous), `ARR_DEL15`, `ARR_DELAY` | Primary modeling targets |
| Delay Cause | `CARRIER_DELAY`, `WEATHER_DELAY`, `NAS_DELAY`, `SECURITY_DELAY`, `LATE_AIRCRAFT_DELAY` | Post-hoc delay attribution (leakage) |
| Taxi / Wheels | `TAXI_OUT`, `TAXI_IN`, `WHEELS_OFF`, `WHEELS_ON` | Ground movement times (post-hoc leakage) |
| Cancellation | `CANCELLED`, `CANCELLATION_CODE`, `DIVERTED` | Flight outcome flags |
| Hourly Weather | `HourlyWindSpeed`, `HourlyVisibility`, `HourlyDryBulbTemperature`, `HourlyPrecipitation`, `HourlyRelativeHumidity`, `HourlyStationPressure`, `HourlyWindGustSpeed` | Real-time weather at departure airport |
| Daily Weather | `DailyAverageWindSpeed`, `DailyPrecipitation`, `DailyMaximumDryBulbTemperature` | Aggregated daily summaries |
| Station Metadata | `STATION`, `ELEVATION`, `station_id` | Weather station identifiers & elevation |

**Handling Data Leakage and Feature Integrity:** A key distinction in our methodology is the strict exclusion of data leakage. Specifically, features from the Actual Performance, Delay Cause, and Taxi/Wheels families are not available at prediction time. Including these features would result in data leakage in our machine learning models, as they represent information that is only known after a flight has already departed or landed. While these fields are highly useful for EDA (Exploratory Data Analysis) to help us understand historical delay patterns, they must be excluded from our modeling features to ensure the integrity and real-world applicability of our predictions.

---
#### Engineered Modeling Feature Families
After data cleaning, leakage exclusion, and feature engineering, the raw columns are transformed into **813 model-ready features** organized into 6 families. All features use only information available **before departure** (2-hour lookahead window).

**Engineered Feature Families (813 Total Columns)**
| Family | Columns | Description | Example Features |
| --- | --- | --- | --- |
| **Temporal & Cyclical** | 10 | Raw time fields plus sin/cos encodings of departure hour and month, binary flags for weekends and peak hours | `dep_hour_sin/cos`, `month_sin/cos`, `MONTH`, `DAY_OF_MONTH`, `dep_hour`, `day_num`, `is_weekend`, `is_peak_hour` |
| **Weather (Raw + Derived)** | 17 | 12 NOAA hourly station observations (temperature, wind, pressure, visibility, precipitation) plus 5 engineered binary thresholds and composite severity index | `HourlyDryBulbTemperature`, `HourlyWindSpeed`, `HourlyVisibility`, `weather_severity`, `low_visibility`, `high_wind` |
| **Historical Delay Patterns** | 19 | Rolling 7d/14d/30d delay rates and flight frequencies per origin and carrier (12) plus target-encoded mean delay rates for origin, dest, carrier, route, and tail (7) | `avg_delay_origin_7d`, `pct_delayed_origin_30d`, `carrier_avg_delay_rate`, `route_avg_delay_rate` |
| **Graph / Network Centrality** | 5 | Directed flight network from training data; PageRank, degree centrality, and route asymmetry capture hub importance and carrier network position | `origin_pagerank`, `dest_pagerank`, `total_degree`, `route_pagerank_diff`, `carrier_pagerank` |
| **Geographic & Flight Info** | \~757 | Distance, elapsed time, flight number (4 numeric) plus one-hot encoding of ORIGIN, DEST, OP\_UNIQUE\_CARRIER, DAY\_OF\_WEEK (\~753 OHE columns) | `DISTANCE`, `CRS_ELAPSED_TIME`, `ORIGIN_ohe`, `DEST_ohe`, `OP_UNIQUE_CARRIER_ohe` |
| **Events & Congestion** | 5 | Static holiday and event flags from public calendars (OPM, NFL, BTS) plus real-time departure volume at origin airport | `is_holiday`, `is_national_event`, `holiday_proximity`, `days_to_holiday`, `origin_hourly_flights` |

**Total:** 60 dense numeric features + \~753 one-hot encoded columns = **813 model input columns**


---

   
## 2. Data Cleaning and Workflow
To get our 5-year dataset (all 31.6 million rows) ready for modeling, we put together a solid data cleaning workflow. Our process focused on removing uninformative data and filling in missing values (nulls) through a multi-stage imputation strategy. By replacing these gaps with smart estimates, we were able to keep our dataset intact while maintaining high data quality. Additionally, we ensured that no "future" information, known as data leakage, was included in the features used to train the machine learning models.

---
#### Step 1: Duplicate Verification
The 5-year dataset, which consists of 31,673,119 rows, contains no exact duplicate rows. Because every record is unique, no deduplication (the process of dropping duplicate data) was required. This confirms the initial integrity of the raw data before we moved into the cleaning and imputation phases.

---
#### Step 2: Null Tier Summary
Before we start the data cleanup of null values, we first must understand the data structure. We categorized the 214 columns from the dataset into five "tiers" based on how much data was missing. This allows us to identify which columns are actually useful for our machine learning model predictions and helps us drop any irrelevant or empty columns. However, for the columns identified as useful that still have some missing data, we need to understand exactly how to handle any null values found in them.

**Overview of Data Completeness and Column Filtering**
| Null Tier | # Columns | Context | Business Logic |
| --- | --- | --- | --- |
| 100% null | 66 | `Monthly*` weather aggregates (e.g. `MonthlyAverageRH`, `MonthlyTotalSnowfall`, `MonthlySeaLevelPressure`) | Weather was joined at the *hourly* level — monthly summaries were never populated |
| 90–99.9% null | 26 | `Daily*` weather summaries (e.g. `DailyAverageSeaLevelPressure`, `DailySnowDepth`, `DailyPeakWindSpeed`, `Sunrise`, `Sunset`) | Daily aggregates reported by only \~0.1–0.2% of station-flight pairs; hourly columns already capture the same signals at finer granularity |
| 50–89.9% null | 18 | Delay cause columns (`CARRIER_DELAY`, `WEATHER_DELAY`, etc. at \~80%), `HourlyWindGustSpeed` (86.9%), `BackupElevation/Lat/Lon` (\~73%), `HourlyPressureChange` (\~66%) | Delay causes only filled for delayed flights (\~18%); wind gusts only logged when they occur; backup station fields sparse by design |
| 1–49.9% null | 31 | `HourlyPrecipitation` (17.0%), `HourlySeaLevelPressure` (\~10%), arrival performance columns (\~2–3%), `WindEquipmentChangeDate` (\~35%) | Station reporting gaps, cancelled flights missing performance data, trace-precipitation parsing failures |
| 0% null | 73 | Temporal (`YEAR`, `MONTH`, `FL_DATE`), carrier/flight ID, route/airport, schedule, distance, station metadata | Core BTS flight record fields — always populated |

**Strategy:** We removed the top two tiers (92 columns) because they were essentially empty. We also excluded the middle tier from our training features because it contains "data leakage"—information like the cause of a delay that you only know after the flight has already happened. Our final, actionable features come from the bottom two tiers, giving us 104 high-quality columns to work with

---
#### Step 3: Null Handling Strategy
Starting from 214 raw columns, we used this 7-step strategy to reduce the dataset to 114 high-quality, clean columns.

**7-Step Data Refinement and Cleaning Workflow**
| Step | Action | Columns Affected | Details |
| --- | --- | --- | --- |
| 1 | Drop uninformative columns | 100 dropped → 114 remain | 92 cols ≥90% null (`Monthly*`, `Daily*`), 9 backup-station metadata cols (64–74% null), 1 non-predictive (`WindEquipmentChangeDate`) |
| 2 | Fill delay-cause nulls with 0 | 5 cols (`CARRIER_DELAY`, `WEATHER_DELAY`, `NAS_DELAY`, `SECURITY_DELAY`, `LATE_AIRCRAFT_DELAY`) | Null = flight was not delayed by that cause; 0 is the true value. These are post-hoc leakage and excluded from features |
| 3 | Cast weather strings → double | 12 cols (e.g. `HourlyVisibility`, `HourlyPrecipitation`, `HourlyWindSpeed`) | Strip NOAA markers (`"T"`, `"s"`), then cast to `double` |
| 4 | Semantic zero-fill for event-based cols | 3 cols | `HourlyWindGustSpeed` (86.9% null) → 0 (no gust observed); `HourlyPresentWeatherType` → `"None"`; `HourlySkyConditions` → `"None"` |
| 5 | 3-stage weather imputation | 13 continuous weather cols (4–66% null) | See detailed breakdown below |
| 6 | Median imputation for flight numerics | 56 cols (0–3% null) | Schedule/route and actual-performance columns filled with column median (robust to right-skewed delay distributions) |
| 7 | Mode imputation for categoricals | 2 of 29 string cols | `TAIL_NUM` and `REM` — the other 27 string columns were already 0% null |

**Strategy for Filling Weather Gaps: 3-Stage Imputation**

To ensure our models have a complete picture of the conditions at takeoff, we used a cascading strategy to fill gaps in our 13 continuous weather observations. This "three-stage" approach moves from the most specific local data to broader averages to ensure every flight is covered.

In Stage 1, we apply a "forward-fill" method within the same airport and date. This essentially means we take the last observation carried forward. For example, if a weather reading is missing for a 2:00 PM flight, we look at the most recent report from that same morning. Since weather usually changes gradually throughout the day, the most recent reading at that specific airport is our best short-term estimate.

If that data isn't available for that day, we move to Stage 2, which uses the "airport-month mean." This captures the typical seasonal weather for that specific location, such as using the average January visibility at O'Hare when a same-day reading is missing.

Finally, if both local options fail, Stage 3 serves as our safety net by using the global median for that variable. We chose the median as our final fallback because it remains stable and is not skewed by the extreme outliers common in weather data, such as heavy storm surges or record-breaking precipitation.

---
#### Step 4 Final Data Verification
**Post-Cleaning Data Integrity and Target Field Status**
| Dataset | Total Rows | Cleaned Columns | Remaining Nulls |
| --- | --- | --- | --- |
| 60M (2015–2019) | 31,673,119 | 114 | `DEP_DELAY`: 475,789 (1.5%), `DEP_DEL15`: 475,789 (1.5%) |

The two columns identified in the table above, `DEP_DELAY` and `DEP_DEL15`, are the core target features that our machine learning models will use as the metric to measure their output. These are the only remaining columns in the entire dataset that contain null values. This is intentional because these nulls correspond to cancelled flights that never actually departed. Since the flight didn't happen, there is no delay value to record. We have intentionally preserved these as nulls so they can be filtered out rather than imputed before model training, as trying to predict a delay for a flight that was cancelled would not be meaningful for the model.

---
#### Step 5: Feature Selection & Modeling Schema
After completing our data analysis, ensuring predictive integrity, and preparing the weather data, we narrowed the dataset down to the most impactful columns. To ensure a clean transition into the modeling pipeline, we perform a final row-level removal of any remaining null values in these specific fields. It is important to note that the only columns with remaining null values are our two target columns, which represent the actual machine learning outputs we are trying to predict.

**Final Feature Catalog and Model Roles**
| Column | Type | Description | Role |
| :--- | :--- | :--- | :--- |
| `MONTH` | int | Month of flight (1–12) | Feature |
| `DAY_OF_WEEK` | int | Day of week (1=Mon, 7=Sun) | Feature |
| `DAY_OF_MONTH` | int | Day of the month (1–31) | Feature |
| `CRS_DEP_TIME` | int | Scheduled departure time (HHMM, local) | Feature (via `DEP_HOUR`) |
| `DEP_HOUR` | int | Derived: `CRS_DEP_TIME / 100` (hour 0–23) | Engineered Feature |
| `CRS_ARR_TIME` | int | Scheduled arrival time (HHMM, local) | Feature |
| `DISTANCE` | double | Flight distance in miles | Feature |
| `CRS_ELAPSED_TIME` | double | Scheduled flight duration in minutes | Feature |
| `OP_CARRIER` | string | Carrier code (14 unique: WN, DL, AA, UA, etc.) | Feature (categorical) |
| `ORIGIN` | string | Departure airport code (369 unique) | Feature (categorical) |
| `DEST` | string | Arrival airport code (368 unique) | Feature (categorical) |
| `DEP_TIME_BLK` | string | Departure time block (e.g., "0800-0859") | Feature (categorical) |
| `HourlyDryBulbTemperature` | double | Dry bulb temperature (°F) | Weather Feature |
| `HourlyDewPointTemperature` | double | Dew point temperature (°F) | Weather Feature |
| `HourlyRelativeHumidity` | double | Relative humidity (%) | Weather Feature |
| `HourlyVisibility` | double | Visibility (statute miles) | Weather Feature |
| `HourlyWindSpeed` | double | Wind speed (mph) | Weather Feature |
| `HourlyWindGustSpeed` | double | Wind gust speed (mph) | Weather Feature |
| `HourlyStationPressure` | double | Station pressure (inHg) | Weather Feature |
| `HourlyPrecipitation` | double | Hourly precipitation (inches) | Weather Feature |
| `DEP_DELAY` | double | Departure delay in minutes (negative = early) | Regression target |
| `DEP_DEL15` | double | Binary: 1 if `DEP_DELAY` ≥ 15 min, else 0 | Classification target |

**Summary of Schema:** Our final selection includes 19 core features and 2 target outputs (`DEP_DELAY` and `DEP_DEL15`) for our classification and regression models. These targets are the actual ML outputs we want to predict. We grouped these into four main categories: Temporal (tracking time patterns), Flight Metrics (distance and duration), Categorical (carrier and route specifics), and Hourly Weather (atmospheric conditions at departure).

**Categorized Modeling Features and Machine Learning Targets**
| Feature Category | Columns |
| :--- | :--- |
| Temporal | `MONTH`, `DAY_OF_WEEK`, `DAY_OF_MONTH`, `DEP_HOUR` (Engineered) |
| Flight Metrics | `DISTANCE`, `CRS_ELAPSED_TIME`, `CRS_ARR_TIME` |
| Categorical | `OP_CARRIER` (14), `ORIGIN` (369), `DEST` (368), `DEP_TIME_BLK` |
| Hourly Weather | `HourlyDryBulbTemperature`, `HourlyDewPointTemperature`, `HourlyRelativeHumidity`, `HourlyVisibility`, `HourlyWindSpeed`, `HourlyWindGustSpeed`, `HourlyStationPressure`, `HourlyPrecipitation` |
| Targets | `DEP_DEL15` (Binary), `DEP_DELAY` (Continuous) |

---
#### Step 6: Summary Statistics & Business Insights
With five years of history, the final dataset is robust enough to handle all weather extremes—from summer storms to winter blizzards—while tracking how carriers perform year over year.

**Flight Delay Metrics:**
| Stat | DEP_DELAY | DEP_DEL15 |
| --- | --- | --- |
| Count | 31,197,330 | 31,197,330 |
| Mean | 9.85 | 0.1819 |
| Std Dev | 43.48 | 0.3858 |
| Min | -234.0 | 0.0 |
| 25th %ile | -5.0 | 0.0 |
| Median | -2.0 | 0.0 |
| 75th %ile | 7.0 | 0.0 |
| Max | 2,755.0 | 1.0 |

**Understanding Our Target Metrics: Classification vs. Regression** 

There is a massive gap between an average flight and extreme cases. Our median delay is -2.0 minutes, meaning over half of all flights depart early. However, the max delay is a staggering 2,755 minutes (nearly two days). This extreme outlier pulls our mean up to 9.85 minutes, even though most flights are not that late. Because the data is so skewed, we use two target columns for our ML outputs.
First, the classification target `DEP_DEL15` is a binary "Yes/No" switch. It does not care if a flight is 20 or 2,000 minutes late; it only flags if it crossed the 15-minute threshold. Since only 18.19% of flights are delayed (a class imbalance), we must be careful. If the model just guesses "On-Time" every time, it would be 82% accurate but totally useless for finding actual delays.
Second, the regression target `DEP_DELAY` predicts the specific number of minutes. This is harder because extreme outliers can throw off the math. By using both, we first identify if a flight will be late, then estimate the severity. This helps us distinguish between a minor 20-minute setback and a major multi-hour disruption.

**5-Year Environmental and Operational Baseline Statistics**
| Feature | Mean | Std Dev | Null % | Notes |
| --- | --- | --- | --- | --- |
| HourlyWindSpeed (mph) | 9.09 | 5.71 | 0.3% | Consistent across years |
| HourlyVisibility (mi) | 9.43 | 1.86 | 0.3% | Generally high; driven down by fog/storm events |
| HourlyDryBulbTemp (°F) | 63.88 | 18.79 | 0.3% | Full seasonal range captured across 5 years |
| HourlyPrecipitation (in) | 0.003 | 0.03 | 17.0% | Mostly zero; right-skewed by storm events |
| HourlyRelativeHumidity (%) | 60.56 | 21.67 | 0.3% | Stable year-round |
| HourlyStationPressure (inHg) | 29.15 | 1.32 | 1.0% | Altitude-dependent, not seasonal |
| HourlyWindGustSpeed (mph) | 25.44 | 5.92 | 86.9% | Only recorded when gusts occur (filled 0 in clean) |
| DISTANCE (mi) | 822.41 | 607.23 | 0.0% | Right-skewed; most flights are short-to-medium-haul |
| ELEVATION (ft) | 249.15 | 397.68 | 0.0% | Airport elevations don't change |

We used a 5-year (2015–2019) window so the model understands long-term patterns, not just a single year. This ensures predictions are built on a stable dataset rather than a short-term snapshot.

---

## 3. EDA
### Flight Delay EDA

---
#### 3.1 Dataset Size and Loading
The dataset for this machine learning workflow is a 60-month combined set from 2015 to 2019, containing 31,673,119 rows and 214 columns. We use 2015–2018 for training and hold out 2019 as a blind test set for model evaluation. The test set contains 7.39M rows, making up about 23.3% of the total data.

**Data Split for Model Training and Evaluation**
| Split | Years | Flights | % of Dataset |
| --- | --- | --- | --- |
| **Train** | 2015–2018 | 24.28M | 76.66% |
| **Test (blind)** | 2019 | 7.39M | 23.34% |

Our primary ML outputs (the target columns) are DEP_DEL15 for classification and DEP_DELAY for regression. The delay rate stays consistent across each year (17.2% to 18.7%), confirming that the 2019 test set is a reliable representation of the overall patterns. The small gap between total flights and "valid" rows represents cancelled flights, which are excluded from the ML targets.

**Yearly Breakdown of Flight Volume and Delay Consistency**
| Year | Total Flights | Valid (non-null) | Delayed | Delay Rate |
| --- | --- | --- | --- | --- |
| 2015 | 5.81M | 5.73M | 1.06M | 18.44% |
| 2016 | 5.61M | 5.55M | 0.95M | 17.16% |
| 2017 | 5.65M | 5.57M | 1.01M | 18.09% |
| 2018 | 7.20M | 7.09M | 1.30M | 18.40% |
| **2019 (test)** | **7.39M** | **7.26M** | **1.36M** | **18.67%** |

---
### 3.2 Input Variable Distributions
#### 3.2.1 Schedule and Calendar Features
We analyzed six schedule and calendar features across the 60-month dataset. The scheduled departure and arrival times(`CRS_DEP_TIME` and `CRS_ARR_TIME`) follow relatively unimodal distributions. The departure time plot ramps up significantly around 5:00 AM to 6:00 AM before leveling off during the midday hours and eventually tapering off for red-eye flights late in the evening. With the mean and median sitting very close together (\~1:30 PM and \~1:25 PM), the distribution is relatively symmetrical. Arrival times follow a nearly identical shape but are shifted roughly 1.5 hours later. This distribution is slightly more spread out and has a heavier right tail due to late-night arrivals.

Scheduled flight time and distance(`CRS_ELAPSED_TIME` and `DISTANCE`) follow extremely similar, heavily right-skewed distributions. While the average flight is 142 minutes and 822 miles, the medians are notably lower at 125 minutes and 650 miles. This confirms that the majority of operations are short-haul flights, while a minority of long-distance trips pull the average upward. Because these two features are highly correlated—longer distances naturally require more time—they likely encode the same predictive signals for the model. 

The day of the month (`DAY_OF_MONTH`) is nearly uniform across the 5-year period. This is expected as airlines generally maintain consistent daily schedules. The visible dips at days 29, 30, and 31 are simply due to months like February having fewer days. Similarly, the day of the week(`DAY_OF_WEEK`) is relatively uniform from Monday through Sunday, with a noticeable dip on Saturdays. This decrease reflects lower demand from business travelers during the weekend.
<img src="https://raw.githubusercontent.com/sacook828-berkeley/261_final_proj_images/main/phase3/schedule_calendar_distributions_60m.png">

#### 3.2.2 Carrier Features
The 60-month dataset captures 19 unique carriers, providing a comprehensive look at the industry's major players.

**Final Carrier Fleet and Codes**
| Code | Airline |
| --- | --- |
| WN | Southwest Airlines |
| DL | Delta Air Lines |
| AA | American Airlines |
| OO | SkyWest Airlines |
| UA | United Airlines |
| EV | ExpressJet Airlines |
| B6 | JetBlue Airways |
| AS | Alaska Airlines |
| MQ | Envoy Air (American Eagle) |
| NK | Spirit Airlines |
| YX | Republic Airways |
| OH | PSA Airlines |
| F9 | Frontier Airlines |
| 9E | Endeavor Air |
| YV | Mesa Airlines |
| HA | Hawaiian Airlines |
| VX | Virgin America |
| G4 | Allegiant Air |
| US | US Airways |

In the 60-month dataset, Southwest Airlines (WN) dominates flight volume with \~21% of all records. Delta (DL, \~15%), American (AA, \~14%), and SkyWest (OO, \~11%) follow closely. This carrier distribution is highly consistent across our 3-month, 12-month, and 60-month datasets. This stability throughout time confirms that while some airports are added or positions shift slightly, the overall distribution shape remains constant.
<img src="https://raw.githubusercontent.com/sacook828-berkeley/261_final_proj_images/main/carrier_3m_distribution.png">
<img src="https://raw.githubusercontent.com/sacook828-berkeley/261_final_proj_images/main/carrier_12m_distribution.png">
<img src="https://raw.githubusercontent.com/sacook828-berkeley/261_final_proj_images/main/phase3/carrier_60m_distribution.png">

#### 3.2.3 Weather Features
Our weather data captures a wide range of atmospheric conditions across the US. Hourly wind speed (`HourlyWindSpeed`) is rightward-skewed, peaking at lower speeds with a tail extending past 30 mph. The mean of 9.1 mph and median of 8.0 mph confirms this skew, which makes sense as most flights depart in relatively calm conditions. Hourly visibility (`HourlyVisibility`) is heavily left-skewed; the overwhelming majority of observations sit at 10 miles (clear conditions), while the small tail below 10 miles represents flights departing in fog or heavy precipitation.

The dry bulb temperature(`HourlyDryBulbTemperature`) shows a roughly normal distribution with a slight left skew (mean \~64°F, median \~66°F). The massive range from −40°F to 110°F reflects the diverse climates in our 5-year dataset. Similarly, the wet bulb temperature (`HourlyWetBulbTemperature`)—the lowest temperature achievable through evaporation—follows a near-identical pattern but is shifted \~9°F cooler. Hourly precipitation (`HourlyPrecipitation`) is almost entirely zero, with an extreme tail reaching 3.5 inches during severe weather events. Hourly relative humidity (`HourlyRelativeHumidity`) is relatively normally distributed but with a left skew and a secondary peak near 100%.

Hourly station pressure (`HourlyStationPressure`), the barometric pressure measured at specific altitudes, shows a leftward-skewed distribution that is likely bimodal. The primary peak sits at \~30 inHg, while a secondary cluster appears around 24 to 26 inHg. This secondary peak represents lower pressure values measured at high-altitude airports like Denver. In contrast, hourly sea level pressure(`HourlySeaLevelPressure`) is a very tight, symmetric distribution centered exactly at 30 inHg. This narrower distribution is expected because it is already elevation-corrected. 

Finally, hourly wind gusts (`HourlyWindGustSpeed`) follow a symmetric distribution with a slight right skew, though this field is only populated for 13% of observations.

<img src="https://raw.githubusercontent.com/sacook828-berkeley/261_final_proj_images/main/phase3/weather_feature_distributions_60m.png">

#### 3.2.4 Airport & Time-Block Features

**Origin Airports:**
The left panel shows the top 20 origin airports. Atlanta dominates at \~7% of all departures, followed by Chicago O'Hare at \~5% and Dallas-Fort Worth at \~4%. The top 5 hubs account for around 25% of all departures. The distribution shape itself has a long tail, the bottom 3 airports in this top 20 sample (DCA, JFK, SLC) hold under \~2% of departing flights. The 12-month vs. 60 month distribution are nearly identical for each airport. 

**Destination Airports:**
The middle panel is almost an exact copy of the left panel, which is expected given that most routes are bidirectional. Atlanta and Chicago O'Hare lead here again. The 12-month and 60-month distributions are also very similar in this visualization.

**Departure Time Block Distribution:**
Flight departures are mostly uniform during the main daily operating hours (7AM-9PM), with each block holding around 7% of departures. The early morning and late night blocks drop significantly to \~3%, reflecting minimal red-eye flight scheduling. Again, the 12-month and 60-month distributions overlap almost perfectly, confirming a stable schedule pattern throughout the full 5-year window.

<img src="https://raw.githubusercontent.com/sacook828-berkeley/261_final_proj_images/main/phase3/categorical_60m.png">

---
### 3.3 Target Variable Analysis 
#### 3.3.1  Target Variable Distribution - Classification: Binary Flag Departure Delay >= 15 Minutes 
The target variable for our classification model is `DEP_DEL15`. This is a binary "Yes/No" flag where 1 indicates a delay of 15 minutes or more. In machine learning terms, these targets are the "ground truth" or the actual outputs the model is trying to predict. This column is naturally imbalanced because approximately 81.8% of flights between 2015 and 2019 were on time. This has a direct impact on how we train the model. If a naive baseline model simply guessed "on-time" every single time, it would reach 81.8% accuracy but fail to catch any actual delays. To handle this, we use *AUC-PR* as our primary evaluation metric (will be discussed further in the Pipeline and Metrics Section). Class-weighting and undersampling are applied in an attempt to handle the imbalanced target variable. The classification distribution plot below confirms the imbalance of delayed vs. on-time flights visually.

<img src="https://raw.githubusercontent.com/sacook828-berkeley/261_final_proj_images/main/phase3/dep15_60m.png">

#### 3.3.2  Target Variable Distribution - Regression: Departure delay in Minutes
When we look at all flights, the departure delay (`DEP_DELAY`) distribution shows a significant right-skew. This means the vast majority of flights actually depart on time or even early. In fact, a large number of flights show a negative delay, which likely reflects the "schedule padding" airlines build in to ensure they stay on track. The distribution drops off sharply after the 0-minute threshold and continues to decrease steadily, which can make it tricky for a model to learn the specific differences right around the 0 to 30-minute boundary. Because the regression data is so skewed by extreme outliers, we use MAPE as our primary evaluation metric to keep our error measurements meaningful (this will be discussed further in the Pipeline and Metrics section).

<img src="https://raw.githubusercontent.com/sacook828-berkeley/261_final_proj_images/main/phase3/dep_delay_60m.png">

---
### 3.4 Seasonal Trends and Performance Analysis
The plots below detail the delay rate and flight volume on a month-to-month basis from 2015 to 2019. By comparing these two metrics, we can see how seasonal pressures affect airline reliability.

**Analysis of Seasonal Delay Rates:**
The left panel illustrates the delay rate and reveals a strong seasonal pattern that repeats across all five years. The summer months (June, July, and August) are consistently the most challenging for the industry. During this period, delays occur between 19% and 24% of the time, eventually peaking at 24% in June 2019. We also see a secondary spike during the winter months, specifically between December and February, where delay rates range from 18% to 22%. While the summer peaks often align with high travel demand, these winter disruptions are likely driven by weather-related issues. Interestingly, the fall months appear to be the most reliable time for travel, with delay rates dropping to their lowest points at 12% to 14%.

**Flight Volume and Capacity Growth:**
The right panel describes total flight volumes and shows two distinct "tiers" of data. From 2015 to 2017, volumes averaged between 450K and 500K flights per month. However, there was a dramatic jump in 2018 and 2019, where volumes increased to between 550K and 650K flights. This shift likely reflects a significant growth in industry capacity. Similar to the delay plots, the summer months show the highest flight volumes each year, signifying the heavy demand of summer vacations. However, a key difference emerges in the winter: flight volumes do not spike in December the way delay rates do.

**Key Findings on Volume vs. Delay:**
Based on these two plots, we can conclude that high delay rates are not purely caused by high flight volumes. For example, December shows very high delay rates despite having relatively average flight volumes compared to the rest of the year. September tells an even more interesting story: it maintains near-peak flight volumes but consistently records the lowest delay rates of the entire year.

<img src="https://raw.githubusercontent.com/sacook828-berkeley/261_final_proj_images/main/phase3/seasonal_delay_60m.png">

---
### 3.5 Carrier and Airport Analysis
To better understand how different airlines perform, we analyzed delays across the industry's major players. These plots examine performance on a carrier-level basis to identify long-term trends and operational shifts.

**Analysis of Carrier Performance Trends (2015–2019):**
The first graph, *Delay Rate Trend*, tracks the percentage of flights delayed for the top ten carriers by volume. Spirit Airlines stands out as the most improved airline in the dataset, successfully dropping its delay rate from \~27% in 2015 to \~18% in 2019. In contrast, Alaska Airlines saw its performance deteriorate significantly during the same period, with delay rates climbing from \~11% to around 19%. Other carriers like JetBlue and Southwest remain consistently high, hovering between 20% and 23%. Delta continues to be the best performer in the group, maintaining steady delay rates around 14% to 15%. United shows more volatility, with a noticeable spike in delays around 2017 before it began to recover.

**Operational Metrics and Industry Volume:**
The middle graph, *Average Departure Delay*, mirrors these trends but expresses them in actual minutes. Most carriers stay within the 8 to 12 minute range for their average delay. Interestingly, Alaska had a near-zero average delay in 2015 before it started to converge with the rest of the pack. Spirit’s improvement is just as visible here as it was in the rate trend chart, showing a clear reduction in the time passengers spent waiting. The final graph, *Flight Volume*, displays each carrier’s share proportional to the total industry volume. During this five-year window, total flight volume grew from \~5M to \~7.5M. Southwest maintains the highest share of flights overall, while American Airlines appears to have grown the most in terms of total scale. This overall increase in flight volume matches the heatmap patterns seen in section 3.4, where the widening of the stack clearly illustrates the industry's rapid growth.

<img src="https://raw.githubusercontent.com/sacook828-berkeley/261_final_proj_images/main/phase3/carrier_level_delay_60m.png">

---
### 3.6 Weather-Delay Correlation Analysis
Twe plots were generated to show how our input features correlate with each other. The right panel looks specifically at the relationship between weather features and `DEP_DELAY`. Across the board, individual weather features show a very weak correlation with departure delays. This matches our earlier delay plots: weather-related delays are relatively rare, and when they do happen, their impact on an individual flight level is minimal when averaged across the entire 31.6 million row dataset.

The left graph breaks down these individual correlations. Features like precipitation(`HourlyPrecipitation`) (+0.055), relative humidity (`HourlyRelativeHumidity`)(+0.054), wind gust speed (`HourlyWindGustSpeed`) (+0.052), wind speed (`HourlyWindSpeed`)(+0.024) all show a positive correlation. This means that as these values increase, we see a marginal increase in delays. On the flip side,visibility (−0.041) have a negative correlation, confirming that clear skies and stable high-pressure systems generally correlate with fewer delays. Other factors like temperature (−0.006), elevation (−0.008), station pressure (+0.002), and wet bulb temp (+0.010) show essentially no correlation at all.

<img src="https://raw.githubusercontent.com/sacook828-berkeley/261_final_proj_images/main/phase3/weather-delay_corr_60m.png">

---
### 3.7 Geographic Delay Analysis
The geographic scatter plot of all 320 origin airports reveals clear spatial clustering among delay-prone locations. Delay rates, calculated as the total number of delayed flights divided by the total number of flights, range from a low of 2.5% to a high of 43.8%, with a median rate of 15.1% across the board.

Major hub airports like ATL, ORD, DFW, DEN, and LAX are represented by the largest bubbles due to their massive flight volumes, yet they maintain delay rates near the national average of 16% to 22%. Interestingly, the highest delay rates are often found at smaller regional airports that have limited capacity and fewer alternatives when bad weather hits. Geographically, the Northeast and Chicago areas show consistently higher delay rates, likely reflecting more unpredictable weather patterns. In contrast, Western airports generally show lower delay rates, which is consistent with more favorable weather and less congested airspace.

This geographic variation shows why including origin airports (`ORIGIN`) and destination airports (`DEST`) as categorical features is so important. Using the airport identity allows the model to capture critical factors like runway capacity, local climate, and airspace congestion that weather observations alone might miss.

<img src="https://raw.githubusercontent.com/sacook828-berkeley/261_final_proj_images/main/phase3/geo_plot_60m_2.png">

---
### 3.8 Key EDA Takeaways and Modeling Implications
The most significant finding from our Exploratory Data Analysis (EDA) is that scheduling and carrier factors dominate delay prediction, while weather plays a minor supporting role. Carrier identity, departure hour, and day of the week appear to have the strongest signals for predicting a delay. One of the biggest drivers we found is "Late-aircraft propagation," where an inbound aircraft’s tardiness cascades directly into the next flight. This is the single largest cause of delays, averaging about 30 minutes among flights that are already delayed.

Weather is a secondary but still important subset of features. Interestingly, all individual weather–delay correlations fall below 0.09. However, this weak linear relationship actually masks the true predictive value of weather. Our plots show that weather features differ between classes primarily in their "tails". This means that while low-visibility events and heavy precipitation are rare, they are disproportionately associated with delays when they do happen. Tree-based models are likely well-suited to capture these non-linear interactions that a standard linear correlation might miss.

The class imbalance in our binary target, `DEP_DEL15`, makes raw accuracy a misleading evaluation metric. Since a "naive" model could just predict all flights to be on time and achieve \~82% accuracy, we cannot rely on accuracy alone to judge our success. 

Finally, geography introduces meaningful variation within the dataset. Airport delay rates range from a low of 2.5% to a high of 43.8%, with clear spatial clustering around congested areas like the Northeast and Chicago. Including `ORIGIN` and `DEST` as categorical features allows us to encode latent factors such as runway capacity, local climate, and airspace density that hourly weather observations alone do not fully capture.

The heavy concentration of on-time and early flights in our `DEP_DELAY` distribution supports the development of a two-stage modeling pipeline. A standalone regression model would be dominated by the massive amount of zero or negative-delay flights, which would blunt its sensitivity to actual delays. Instead, a classification stage will first separate on-time flights from delayed ones, followed by a downstream regression model to predict the severity of the delay. This strategy will be explored in the coming sections.

# Planned Machine Learning Algorithms and Metrics


## Machine Learning Pipeline

This pipeline predicts flight delays in two stages. Stage 1 is a binary classification task that predicts `DEP_DEL15`, which indicates whether a flight departed 15 or more minutes late (1 = delayed, 0 = on-time). Stage 2 is a regression task that predicts `DEP_DELAY`, the actual departure delay measured in minutes. Together, these stages answer both "Will this flight be delayed?" and "By how many minutes?"

### Data Ingestion and Temporal Split

The pipeline starts with the full 60-month OTPW dataset (January 2015 to December 2019), which contains 31,673,119 raw flight records joined with hourly NOAA weather observations. Before any modeling, we apply two critical filters:

1. **Remove cancelled flights.** Cancelled flights have no departure delay to predict because they never departed. These records have null values in both `DEP_DEL15` and `DEP_DELAY`.
2. **Remove records with null targets.** A small number of non-cancelled flights are also missing delay data due to reporting gaps.

This filtering removes approximately 475,789 rows (1.5%), leaving 31,197,330 ML-ready flight records, each with valid departure delay labels that our models can learn from.

| Dataset Stage | Rows | Description |
| --- | --- | --- |
| Raw OTPW 60M | 31,673,119 | All flight records 2015 to 2019 joined with weather |
| After filtering | 31,197,330 | Cancelled flights and null-target records removed |
| Difference | \~475,789 (1.5%) | Unusable for supervised ML (no labels) |

From these 31.2M machine learning model ready flights, we retain only 34 pre-departure features: schedule identifiers, carrier and aircraft identifiers (`TAIL_NUM`), and 12 hourly weather observations at the origin airport. All post-departure columns (`ARR_DELAY`, `TAXI_OUT`, `CARRIER_DELAY`, `WEATHER_DELAY`, etc.) are excluded to prevent data leakage.

---

#### Temporal Train/Test Split

The data is split temporally using a year-based holdout. All 48 months of 2015 to 2018 serve as training data (\~23.9M rows), and all 12 months of 2019 serve as the blind test set (\~7.3M rows). This mirrors a real operational deployment where the model trains on historical data and evaluates on future, unseen flights.

| Split | Years | Months | Rows | Purpose |
| --- | --- | --- | --- | --- |
| Training | 2015 to 2018 | All 48 months | \~23,933,364 | Model fitting and cross-validation |
| Blind Test | 2019 | All 12 months | \~7,263,966 | Final evaluation simulating production deployment |
| Total | 2015 to 2019 | All 60 months | 31,197,330 | ML-ready records |

**Per-year breakdown:**

| Year | Rows | Delay Rate | Split |
| --- | --- | --- | --- |
| 2015 | 5,726,181 | 18.44% | Train |
| 2016 | 5,546,861 | 17.16% | Train |
| 2017 | 5,572,592 | 18.09% | Train |
| 2018 | 7,087,730 | 18.40% | Train |
| 2019 | 7,263,966 | 18.67% | Test |

---

#### Why This Split Matters for Our ML Workflow

An earlier approach split by quarter (Q1 to Q3 train, Q4 test), but this leaked 2019 data into training because January through August 2019 appeared in the training set, violating temporal holdout. Year-based splitting ensures the blind test represents a genuinely unseen future period, which is exactly what a production model faces.

The delay rate spread across each year is relatively tight (17.2% to 18.7%), which confirms two important things for our ML workflow:

1. **The 2019 test set is representative.** The delay distribution in our held-out year matches the training years, so we are evaluating the model on data that comes from the same underlying process.
2. **Performance reflects prediction difficulty, not drift.** If the model struggles on 2019 data, we can be confident it is because flight delays are inherently hard to predict from pre-departure features, not because the 2019 delay patterns are fundamentally different from 2015 to 2018.

This temporal integrity is critical for the entire downstream pipeline. The expanding-window cross-validation within training (2015 to 2016, 2015 to 2017, 2015 to 2018), the Stage 1 classifier evaluation, and the Stage 2 regression evaluation all depend on this strict separation between past (training) and future (test) data.

### Preprocessing

Weather columns arrive as strings with NOAA quality flags (for example, "0.08V0.08" or trace markers "T"). We use regex to safely convert these to numeric values, returning NULL for unparseable entries. Null imputation follows a three-stage cascading strategy that moves from the most specific local data to broader averages to ensure every flight is covered:

1. **Forward-fill within airport and date.** We take the most recent observation carried forward from that same airport and day. For example, if a weather reading is missing for a 2:00 PM flight, we look at the most recent report from that same morning. Since weather usually changes gradually throughout the day, the most recent reading at that specific airport is our best short-term estimate.
2. **Airport-month mean.** If same-day data is not available, we use the typical seasonal weather for that specific location. For example, we use the average January visibility at O'Hare when a same-day reading is missing.
3. **Global median.** Our final safety net uses the median value for that variable across all observations. We chose the median rather than the mean as our fallback because it remains stable and is not skewed by the extreme outliers common in weather data, such as heavy storm surges or record-breaking precipitation.

#### Categorical Encoding Pipeline

Categorical columns (`OP_UNIQUE_CARRIER`, `ORIGIN`, `DEST`, `DEP_TIME_BLK`) cannot be fed directly into machine learning models as raw text strings because models require numeric input. We convert these text categories into numbers through a four-stage Spark ML transformation pipeline:

**Step 1: StringIndexer (Text to Integer IDs).** Each unique category string is mapped to an integer based on how frequently it appears. For example, the most common carrier Southwest ("WN") becomes 0, Delta ("DL") becomes 1, American ("AA") becomes 2, and so on through to the least frequent carrier at index 18. This is purely a labeling step, and the numbers themselves have no mathematical meaning. Southwest being "0" does not mean it is less than Delta at "1."

**Step 2: One-Hot Encoding (Integer IDs to Binary Indicators).** This is where the dimensionality expansion happens. Each integer ID from Step 1 is converted into a vector of 0s and 1s, where exactly one position is set to 1 ("hot") and the rest are 0. For example, if there are 19 carriers, Southwest becomes [1, 0, 0, ..., 0] (19 positions) and Delta becomes [0, 1, 0, ..., 0]. We do this because without one-hot encoding, the model would treat the integer IDs as ordered values and would assume carrier "18" is somehow "larger" or "more" than carrier "0," which is meaningless.

Spark's `OneHotEncoder` also uses a setting called `dropLast=True` by default, which drops one indicator per categorical column to avoid multicollinearity (a statistical redundancy where one feature is perfectly predictable from the others). For example, if you know all 18 other carrier indicators are 0, you can automatically infer the 19th carrier, so that last column is redundant and gets dropped.

After one-hot encoding all four categorical columns with `dropLast`, the total comes to approximately 753 binary indicator features. The exact count depends on the number of unique values present in the filtered training data (some airports may not appear after removing cancelled flights):

| Column | Unique Values | Binary Columns Created |
| --- | --- | --- |
| `ORIGIN` (departure airport) | \~362 | \~361 (one dropped per `dropLast`) |
| `DEST` (arrival airport) | \~362 | \~361 |
| `DEP_TIME_BLK` (departure hour block) | 19 | 18 |
| `OP_UNIQUE_CARRIER` (airline) | 19 | 18 |
| Total | | \~753 binary indicator columns |

This produces approximately 753 new binary features from just 4 original text columns. That sounds like a lot, but each individual flight only activates 4 of these 753 positions (one per category: one origin, one destination, one carrier, one time block). The other 749 positions are all zeros. Spark stores this efficiently as a sparse vector, recording only the non-zero positions rather than all 753 values. This is what we mean by "sparse dimensions." It uses roughly 15 times less memory than storing every zero explicitly.

**Potential concern: does 753 dimensions cause problems?** In some settings, having far more categorical features (753) than numeric features (60) can cause models to overweight categorical patterns. However, with 31.2 million training rows we have more than enough data to support this dimensionality without overfitting. Tree-based models like GBT are naturally resistant to irrelevant sparse features because they only split on informative columns.

**Step 3: VectorAssembler (Merge All Features).** This step simply combines the 60 numeric features and the approximately 753 one-hot encoded binary features into a single input array per flight. Machine learning models in Spark ML require all features to be in one column (a "feature vector"), so the VectorAssembler stacks them together into an approximately 813-dimensional feature vector (60 + 753).

**Step 4: StandardScaler (Normalize Scale).** Different features have very different numeric ranges. `DISTANCE` ranges from 0 to 5,000 miles, while `is_weekend` is just 0 or 1. Without normalization, features with larger ranges would dominate the model simply because of their scale, not because they are more predictive. The StandardScaler adjusts each feature so they all have similar variance. We deliberately skip mean-centering because subtracting the mean from our sparse vector would fill in every zero entry with a non-zero value, destroying the memory-efficient sparse format and causing out-of-memory errors on our 16 GB workers.

The entire pipeline is fit on the training set only. The fitted pipeline then transforms the test set using the training-derived indices and scaling parameters, preventing any test-set information from leaking into the encoding.

---

### Feature Engineering

The feature space evolves through four stages, culminating in an approximately 813-dimensional sparse vector with approximately 54 non-zero values per row, achieving 15 times memory savings compared to a dense representation:

| Stage | What happens | Feature count |
| --- | --- | --- |
| Raw dataset | 214 columns loaded from OTPW 60M | 214 |
| Column selection | Drop post-departure leakage, 100%-null monthly weather, metadata; keep only pre-departure features and 2 targets | 34 (19 features + 2 targets + schedule/ID cols) |
| Feature engineering | Create cyclical time encodings, weather composites, rolling congestion rates, RFM multi-timeframe features, target-encoded delay rates, graph-based network features, event-based holiday/disruption features | 60 numeric features (19 original + 41 engineered) |
| One-hot encoding expansion | StringIndexer to One-Hot Encoding on 4 categorical columns (\~753 sparse binary cols) | \~813-dim feature vector (60 numeric + \~753 one-hot) |

**Note:** The model receives an approximately 813-dimensional feature vector, but only 41 features are truly engineered. The remaining approximately 753 dimensions are a mechanical one-hot expansion of 4 categorical columns (with `dropLast`, as described in Preprocessing). These encode the same information as the original 4 string columns in a format consumable by Spark ML.

The 60 numeric features break down into nine families:

| Family | Count | Features | Rationale |
| --- | --- | --- | --- |
| Schedule / Calendar | 7 | `CRS_DEP_TIME`, `CRS_ARR_TIME`, `CRS_ELAPSED_TIME`, `DISTANCE`, `DISTANCE_GROUP`, `DAY_OF_MONTH`, `DAY_OF_WEEK` | Core flight schedule attributes available at booking time |
| Raw Weather | 12 | `HourlyVisibility`, `HourlyWindSpeed`, `HourlyDryBulbTemperature`, `HourlyPrecipitation`, `HourlyRelativeHumidity`, `HourlyStationPressure`, `HourlyWindGustSpeed`, `HourlyDewPointTemperature` | Hourly NOAA observations at origin airport, cleaned via safe numeric conversion and three-stage imputation |
| Engineered Time | 8 | `dep_hour`, `hour_sin/cos`, `month_sin/cos`, `day_of_week_sin/cos`, `is_weekend`, `day_num` | Cyclical encodings ensure that 11 PM is treated as close to midnight and December is treated as close to January, rather than being maximally distant values. `day_num` (days since epoch) captures long-term temporal ordering |
| Weather Composites | 5 | `origin_weather_severity`, `dest_weather_severity`, `weather_diff`, `has_precipitation`, `low_visibility` | Combined severity indicators. Individual weather features are weak (absolute r < 0.09 from our EDA) but composites capture multi-factor adverse conditions that single variables miss |
| Congestion and Rolling | 4 | `origin_hourly_flights`, `dest_hourly_flights`, `route_daily_flights`, `tail_rolling_delay_rate` | Same-day rolling averages via window functions. `tail_rolling_delay_rate` captures the late-aircraft cascade effect, which our EDA identified as the single largest delay cause (\~30 min average among delayed flights) |
| Target Encoding | 4 | `carrier_te`, `origin_te`, `dest_te`, `route_delay_rate` | Mean delay rate per entity, computed on train only and joined to test. Captures delay propensity without creating hundreds of additional one-hot columns |
| Multi-Timeframe RFM | 12 | 7d/14d/30d rolling: `carrier_delay_rate`, `origin_delay_rate`, `dest_delay_rate`, `route_delay_rate` | Recency, Frequency, Monetary features across three horizons. 7-day detects short-term disruptions (storms at ORD, crew shortages), 14-day captures biweekly operational patterns and recovery cycles, 30-day reflects seasonal/monthly trends (summer thunderstorms, holiday surges) |
| Graph-based | 5 | `origin_pagerank`, `dest_pagerank`, `total_degree`, `route_pagerank_diff`, `carrier_pagerank` | Network centrality features from a directed flight graph where airports are nodes and routes are edges (weighted by flight count). Captures hub importance and how delays cascade from high-PageRank airports to downstream flights |
| Event-based | 3 | `is_holiday`, `is_national_event`, `holiday_proximity` | Binary flags for US federal holidays (plus or minus 1 day window), major events (Super Bowl weekend, Spring Break peaks), and days-to-nearest-holiday (0 to 7 scale) for pre/post holiday surge |
| **Total numeric** | **60** | 19 original + 41 engineered | |

All 60 numeric and approximately 753 one-hot encoded features are assembled via the VectorAssembler and scaled with the StandardScaler (as described in the Preprocessing section), yielding the final approximately 813-dimension input vector.

#### Graph Feature Implementation: PageRank on the Flight Network

We treat the US air traffic system as a directed graph where each airport is a node and each flight route is a directed edge pointing from the origin airport to the destination. Edges are weighted by the number of flights on that route. A route like ATL to LAX with 50,000 annual flights has a much heavier edge than a regional route with 500 flights.

PageRank quantifies each airport's structural importance based on how many flights arrive from other important airports. All graph computations use training years only (2015 to 2018) to prevent test-set leakage.

The PageRank results revealed clear network structure in the US aviation system. Major hubs like ATL, ORD, DFW, and LAX received the highest scores, reflecting their role as central nodes that receive flights from nearly every other airport. Regional airports received much lower scores. We also computed carrier-specific PageRank because different airlines have different network structures. Southwest operates a point-to-point network where many airports have similar importance, while legacy carriers like Delta have a strong centralized network structure concentrated around ATL and MSP.

The five graph features produced are:

| Feature | What It Captures |
| --- | --- |
| `origin_pagerank` | Importance of the departure airport in the overall national flight network |
| `dest_pagerank` | Importance of the arrival airport in the overall national flight network |
| `total_degree` | Total number of unique routes (incoming + outgoing) at the origin airport, a raw measure of connectivity |
| `route_pagerank_diff` | Difference between origin and destination PageRank, captures directional flow (flights from major airports to smaller airports vs. the reverse) |
| `carrier_pagerank` | Importance of the origin airport within that specific carrier's sub-network (for example, ATL is critical for Delta but less so for Southwest) |

#### Event Feature Data Sources

The holiday and event dates are manually curated lookup tables derived from the following public reference sources:

| Data | Source | Reference |
| --- | --- | --- |
| US Federal Holidays (2015 to 2019) | U.S. Office of Personnel Management (OPM), official observed federal holiday schedule, including Saturday-to-Friday and Sunday-to-Monday observance shifts | [opm.gov/policy-data-oversight/pay-leave/federal-holidays](https://www.opm.gov/policy-data-oversight/pay-leave/federal-holidays/) |
| Super Bowl Dates (XLIX to LIII) | NFL historical records via Pro Football Reference | [pro-football-reference.com/super-bowl](https://www.pro-football-reference.com/super-bowl/) |
| Spring Break Windows | Approximate mid-March peak travel windows based on common US school district calendars | Derived from publicly available K to 12 academic calendars for large districts (NYC DOE, LAUSD, CPS) |

Columbus Day, Veterans Day, and Juneteenth are excluded because they show minimal impact on air travel delay rates based on BTS data patterns.

---

### Checkpoint Strategy

Checkpointing is central to the pipeline's ability to run on our resource-constrained cluster. After preprocessing, datasets are checkpointed to break the Spark DAG lineage. Without this, the DAG grows through feature engineering, assembly, and model training. When Spark re-computes the full lineage during shuffles, the 16 GB workers run out of memory. Checkpointed data is materialized to DBFS reliable storage, so downstream cells read from disk rather than re-executing the entire chain.

| Checkpoint | Contents | Purpose |
| --- | --- | --- |
| Checkpoint 1 | Cleaned and engineered data as Parquet (train + test), plus the fitted transformation pipeline | Resume from feature engineering without re-running upstream cells; break Spark DAG to avoid OOM |
| Checkpoint 1 (Enhanced) | Enhanced data with graph and event features as Parquet; re-split by year (2015 to 2018 train, 2019 test) | Resume from enhanced feature engineering; corrects earlier quarter-based split to year-based |
| Checkpoint 2 | Stage 1 classification models and results CSVs | Resume from evaluation without re-training classifiers |
| Checkpoint 3 | Stage 2 regression models and metrics | Full pipeline artifact storage |
| Pipeline Model | The four-stage transformation pipeline (text-to-integer conversion, one-hot encoding, feature merging, scaling) saved so new data can be processed identically without refitting | Reusable transform pipeline for new data |

This strategy enables resuming from any stage without re-running the full pipeline, which is critical given the multi-hour total runtime on our cluster.

---

### Compute Environment

| Component | Specification |
| --- | --- |
| Cluster | Team_1_1 |
| Driver | m6g.4xlarge (16 vCPU, 64 GB RAM) |
| Workers | 7 x m6g.xlarge (4 vCPU, 16 GB RAM each) |
| Runtime | Databricks 17.3 LTS |
| Memory optimization | Sparse vectors (`withMean=False`), `maxMemoryInMB=512`, `repartition(400)` to prevent OOM during GBT findBestSplits on 60M scale |

---

### Temporal Cross-Validation Strategy

Within the training data (2015 to 2018), we use expanding window cross-validation to validate temporal stability. Each fold trains on all preceding years and validates on the next, mimicking how a production model would be updated annually:

| Fold | Training Years | Validation Year | Train Rows | Val Rows |
| --- | --- | --- | --- | --- |
| Fold 1 | 2015 | 2016 | 5.7M | 5.5M |
| Fold 2 | 2015 to 2016 | 2017 | 11.3M | 5.6M |
| Fold 3 | 2015 to 2017 | 2018 | 16.8M | 7.1M |
| Final | 2015 to 2018 | 2019 (blind) | 23.9M | 7.3M |

This approach prevents future data leakage because the model only ever sees past data during training. It also enables drift detection by showing how model performance evolves as the training window grows. If the model degrades significantly from fold to fold, it signals that the underlying delay patterns are shifting.

---

### Stage 1: Binary Classification

Stage 1 predicts whether a flight will depart at least 15 minutes late using the column `DEP_DEL15` (1 = departure delay of 15 minutes or more, 0 = on-time). The training set has a 4.3:1 class imbalance (81.8% on-time vs 18.2% delayed).

**Class Imbalance Handling: Two Strategies Compared**

We train both models under two class-balancing strategies and compare results head-to-head:

| Strategy | How it works | Applied to | Training set size |
| --- | --- | --- | --- |
| Class Weighting | Delayed flights receive 4.3x weight in the loss function via `weightCol` | GBT (native support) | \~23.9M (full) |
| 1:1 Undersampling | Majority class randomly downsampled to match minority count; training folds only, validation stays at natural distribution | GBT, MLP | \~8.6M (balanced, \~4.3M per class) |

Undersampling reduces the training set by approximately 64%, which also significantly reduces training time (for example, GBT: \~178 min to \~96 min). For MLP, undersampling is the only option because MLlib's `MultilayerPerceptronClassifier` does not natively support `weightCol`.

**Tuning Strategy:** We use an expanding-window temporal CV. Fold 1 trains on 2015 and validates on 2016. Fold 2 trains on 2015 to 2016 and validates on 2017. Fold 3 trains on 2015 to 2017 and validates on 2018. This approach uses temporal ordering rather than random k-fold cross-validation, ensuring the model never trains on future data. The best configuration per model is retrained on all 2015 to 2018 data before blind evaluation on 2019.

#### Model 1: Gradient-Boosted Trees (GBT) (Baseline)

GBT serves as our baseline model for both classification and regression. It optimizes log-loss sequentially, fitting each new tree to the negative gradient (pseudo-residuals) of the loss from previous trees:

$$\mathcal{L}_{GBT} = -\sum_{i=1}^{n}\left[y_i \log(\hat{p}_i) + (1 - y_i)\log(1 - \hat{p}_i)\right]$$

$$f_{m}(x) = f_{m-1}(x) + \eta \cdot h_m(x)$$

where **η** is the learning rate (`stepSize`) and **h_m** is the tree fit to the gradient at iteration **m**.

**Baseline Grid:** `maxIter` in {20, 50}, `maxDepth` = 5, `subsamplingRate` = 0.8 = 2 configurations

**GBT Optimized Grid (Extended):** The GBT Optimized notebook pushes deeper with `maxIter` in {50, 100, 150, 200, 250}, `maxDepth` in {5, 6, 7, 8}, `stepSize` in {0.02, 0.03, 0.05, 0.1}, `subsamplingRate` in {0.7, 0.8} = 8 configurations per variant. Prior tuning identified `maxIter=100, maxDepth=7, stepSize=0.1, subsamplingRate=0.8` as the winning configuration.

**Classification Threshold Tuning:** In GBT, the threshold determines how raw prediction probabilities are converted to binary class labels. The default threshold of 0.5 predicts class 1 if P(delayed) is greater than or equal to 0.5. Lowering the threshold (for example, 0.3) increases recall by catching more delays but may reduce precision, while raising it (for example, 0.65) does the opposite. We test 16 thresholds from 0.20 to 0.65 on validation data to optimize F1 score, then apply the optimal threshold to the blind test. This allows operators at different airports to select their own precision/recall operating point based on local cost structures.

#### Model 2: Multilayer Perceptron (MLP) Neural Networks

A Multilayer Perceptron (MLP) is a type of artificial neural network that takes a fundamentally different approach from the tree-based GBT baseline. While GBT makes predictions by applying a series of yes/no questions about individual features (such as "Is wind speed > 20 mph?" or "Is the carrier Southwest?"), an MLP learns by passing all features simultaneously through layers of interconnected "neurons." Each neuron applies a weighted sum of its inputs followed by a nonlinear activation function, and the network learns the optimal weights through a process called backpropagation, iteratively adjusting weights to minimize prediction error on the training data.

The key difference from GBT:

* **GBT** builds many decision trees, where each tree asks a series of yes/no questions about individual features. This model is excellent at capturing threshold effects and can handle our sparse 813-dimensional input efficiently, but it evaluates features one at a time at each tree split and may miss interactions between features that only matter in combination.
* **MLP neural networks** process all 813 features simultaneously through hidden layers, where each neuron can learn to combine multiple features in nonlinear ways. For example, an MLP might learn that "high wind speed + low visibility + hub airport + evening departure" together indicate a high delay risk, even though none of these features alone is strongly predictive. The tradeoff is that MLPs are harder to interpret (we cannot easily see which feature combinations drive predictions), require more careful tuning, and do not natively handle class imbalance the way tree models do.

We evaluate three architectures using PySpark's `MultilayerPerceptronClassifier` with warm-start early stopping (patience=3, epoch step=10, max 150 iterations):

**Architecture A (Shallow)**
```
Input (813) → 64 → 2 (output)
```
A single hidden layer tests whether a relatively simple nonlinear boundary can capture the classification signal. With approximately 52,160 trainable weights, this is our most parameter-efficient neural model.

**Architecture B (Wider Shallow)**
```
Input (813) → 128 → 2 (output)
```
Doubles the hidden width to test whether additional capacity improves discrimination. With approximately 104,322 weights, this architecture can capture broader feature interactions.

**Architecture C (Deeper)**
```
Input (813) → 128 → 64 → 2 (output)
```
The deep narrowing structure forces the network to learn hierarchical feature interactions. The 813 to 128 compression captures the most informative feature combinations, then 128 to 64 distills decision-relevant patterns. With approximately 112,384 weights, this is our most expressive model.

**MLP Classification Hyperparameters:**

| Parameter | Value | Notes |
| --- | --- | --- |
| `blockSize` | 256 | Mini-batch size; larger means faster but more memory |
| `solver` | `l-bfgs` | Preferred for MLlib MLP; converges faster than gradient descent |
| `tol` | 1e-6 | Convergence tolerance |
| `maxIter` | 150 (with early stopping) | External early stopping monitors validation AUC-PR |

**Total Stage 1 experiments:** 8 extended GBT configs + 16-threshold grid search + 3 MLP architectures x 3 expanding-window folds + MLP-D (No OHE comparison experiment) = approximately 22 classification model fits + final retrains.

---

### Stage 2: Regression

Flights classified as delayed by Stage 1 are fed into Stage 2, which predicts the delay duration in minutes using `DEP_DELAY` as the regression target. This connected pipeline mirrors how the system would work in production: the classifier first decides whether to trigger delay protocols, then the regressor estimates the duration so operations teams can plan gate reassignments, crew swaps, and passenger rebooking with a concrete time estimate.

#### Two Training Approaches

A key design question is what data Stage 2 should train on. We compare two approaches:

| Approach | Training Data | Test Data | Rationale |
| --- | --- | --- | --- |
| A (Ground Truth) | All flights with `DEP_DEL15 = 1` (\~4.4M rows) | Both: (1) Ground-truth `DEP_DEL15 = 1` (\~1.36M, upper bound), (2) Classifier pred=1 AND `DEP_DEL15 = 1` (\~889K, production-realistic) | Highest quality training signal; regression learns from all flights delayed 15 min or more |
| B (Fully Connected) | Out-of-fold classifier predictions where pred=1 AND `DEP_DEL15 = 1` (\~2.9M rows) | Classifier pred=1 AND `DEP_DEL15 = 1` (\~889K) | Matches production noise level; regression sees the same classifier errors it will face at test time |

Approach B uses out-of-fold (OOF) predictions to avoid a subtle but important problem: in-sample bias. If we simply ran the Stage 1 classifier on the same data it was trained on, it would "remember" those examples and produce overly confident predictions, flagging delays with near-perfect accuracy that it could never replicate on new flights. The regression model would then learn to expect these clean, optimistic inputs, only to encounter much noisier classifier outputs at test time.

To fix this, we use a leave-one-year-out rotation strategy. For each training year, we train a fresh GBT classifier on the other three years and predict on the held-out year. Because each classifier has never seen the data it is predicting on, the resulting predictions are honest. They contain the same kinds of mistakes (missed delays, false alarms) that the classifier will make on unseen 2019 flights. After rotating through all 4 years, every training example has an out-of-sample prediction attached to it.

All regression data is filtered to `DEP_DEL15 = 1`, which means only flights that were actually delayed by 15 minutes or more are used for training and evaluation. This filter also removes false positives from the classifier, which are flights the model predicted as delayed but that actually departed on time. Because `DEP_DEL15 = 1` requires a delay of at least 15 minutes, the regression target `DEP_DELAY` ranges from 15 to 2,755 minutes.

#### Label Clipping Strategies

Flight delay distributions are heavily right-skewed: most delays are 15 to 60 minutes, but extreme outliers reach 2,755 minutes (nearly two days). These outliers disproportionately inflate RMSE and can distort the learned model. We compare two clipping strategies:

| Variant | Threshold | Mean | Median | Max | Rows Clipped | Purpose |
| --- | --- | --- | --- | --- | --- | --- |
| Clip@P99 | 300 min (GBT) / 359 min (MLP) | 63.8 min | 41.0 min | 300 to 359 min | \~1.0% | Removes only the most extreme tail while preserving 99% of the delay distribution |
| No Clip | None | 66.1 min | 41.0 min | 2,755 min | 0 | Full distribution retained; tests model robustness to extreme outliers |

P99 clipping caps the extreme tail end of very large delays so they do not skew the model. Delays beyond 5 hours are usually treated as cancellations by operations teams, so predicting exact minutes at that point is not very useful. The exact P99 threshold differs slightly between the GBT notebook (300 minutes) and the MLP notebook (359 minutes) because P99 was recomputed independently on each notebook's filtered regression training set.

#### Regression Models and Loss Functions

We train two regression model families for each approach/variant combination:

**Gradient-Boosted Trees Regressor** sequentially fits trees to the residuals of previous iterations, minimizing squared error:

$$\mathcal{L}_{GBT} = \sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

$$f_{m}(x) = f_{m-1}(x) + \eta \cdot h_m(x)$$

**Baseline Grid:** `maxIter` in {20, 50}, `maxDepth` = 5, `stepSize` = 0.1 = 2 configurations

**GBT Optimized Grid (Extended):** Adds a log-transform variant where we train on `log(delay + 1)` to compress the heavy right tail. This helps GBT allocate tree splits more evenly across the delay range rather than being dominated by extreme outliers. Predictions are converted back to minutes via `exp(prediction) - 1` for evaluation. Grid includes `maxIter` up to 250, `maxDepth` up to 8, `stepSize` as low as 0.02 = 8 configurations per variant.

**MLP Neural Networks (PyTorch Regression):** Since MLlib does not provide a native MLP regressor, we implement true continuous regression using PyTorch on the driver node. Unlike the classification MLPs above (which output a probability for two classes via PySpark's distributed `MultilayerPerceptronClassifier`), these regression MLPs output a single continuous number, the predicted delay in minutes. The training data is converted from Spark ML sparse vectors to scipy CSR (Compressed Sparse Row) matrices, then fed through PyTorch MLP models trained with MSE (Mean Squared Error) loss. This represents an additional deep learning contribution beyond the distributed classification MLPs, extending our neural network approach to continuous regression.

**MLP-A (Shallow Regression):** `Input (813) → 64 → 1`
**MLP-B (Deeper Regression):** `Input (813) → 128 → 64 → 1`

**PyTorch Regression MLP Hyperparameters:**

| Parameter | Value | Notes |
| --- | --- | --- |
| `batch_size` | 4,096 | Larger batches for stable gradient estimates on continuous targets |
| Learning rate | 1e-3 | Adam optimizer |
| Loss function | MSE | True regression minimizes squared difference between predicted and actual delay minutes |
| `patience` | 3 | Early stopping on validation RMSE |
| `max_epochs` | 150 | Maximum training iterations |
| Device | CPU (driver node) | Sparse vectors converted to scipy CSR, then trained on the driver's CPU |

Both architectures use early stopping (patience=3) monitoring validation RMSE. The deeper architecture has greater capacity to learn complex delay duration patterns, while the shallow version tests whether a simpler mapping suffices.

#### Regression Evaluation Metrics

**MAPE** (Mean Absolute Percentage Error) is the primary decision metric for Stage 2. It expresses prediction error as a percentage of the actual delay, providing a scale-invariant measure that lets us compare model quality across different delay magnitudes:

$$MAPE = \frac{100}{n}\sum_{i=1}^{n}\left|\frac{y_i - \hat{y}_i}{y_i}\right|$$

MAPE values of 60 to 90% across models indicate that predictions are typically off by a proportion comparable to the actual delay magnitude, reflecting the inherent difficulty of predicting delay duration from pre-departure features alone. The high MAPE is driven by the combination of (a) many short delays (median = 41 minutes) where even moderate absolute errors produce large percentage errors, and (b) the fundamental unpredictability of delay duration from features available 2 hours before departure.

We report **MAE** (Mean Absolute Error) as a secondary metric because it directly answers the operational question: on average, how many minutes off is the predicted delay from the actual delay?

$$MAE = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|$$

We also track **RMSE** (Root Mean Squared Error), which is particularly important for flight delay prediction because it penalizes large errors disproportionately through the squared term:

$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

This matters operationally because not all prediction errors are equally costly. Underestimating a 3-hour delay by 90 minutes is far more disruptive than underestimating a 30-minute delay by 10 minutes. The first causes missed crew swaps and stranded connections, while the second is a minor inconvenience. RMSE surfaces exactly these high-stakes errors that MAE would average away. It is also the metric that Spark's `RegressionEvaluator` uses by default, making it the natural choice for hyperparameter selection during cross-validation. In our results, RMSE drives model selection (picking the best hyperparameter configuration), while MAPE drives the final model comparison (picking the best approach and variant).

#### Experiment Scale

The full Stage 2 regression pipeline runs: 2 label variants x 2 approaches x 2 model families x (2 to 8 configs per model) = approximately 27 hyperparameter configurations, each evaluated on both connected (production-realistic) and ground-truth (upper bound) test sets.

Combined with Stage 1's approximately 22 classification model fits, the full two-stage pipeline encompasses approximately 48 total experiments across classification and regression, trained on the full 60-month dataset of 31.2 million flights.

<img src="https://raw.githubusercontent.com/sacook828-berkeley/261_final_proj_images/main/phase3/final_pipeline_pic.png">



### Modeling Pipelines

#### Pipeline Visualization

The diagram above shows the full two-stage pipeline architecture. Data flows through four sub-pipelines:

1. **Data Ingestion Sub-pipeline:** Load OTPW 60M, filter cancelled/null-target flights, year-based train/test split
2. **Feature Engineering Sub-pipeline:** Raw features, weather imputation, cyclical encoding, rolling aggregates, target encoding, graph PageRank, event flags, VectorAssembler, StandardScaler, resulting in an approximately 813-dim sparse vector
3. **Stage 1 Classification Sub-pipeline:** Balanced training data, model training (MLP or GBT), expanding-window temporal CV, early stopping, threshold optimization, predict `DEP_DEL15`
4. **Stage 2 Regression Sub-pipeline:** Connected subset (predicted delayed intersected with actually delayed), out-of-fold data preparation, model training (PyTorch MLP or Spark GBT), target clipping/log transform, predict `DEP_DELAY` minutes

#### Families of Input Features

| Family | Count | Examples | Rationale |
| --- | ---: | --- | --- |
| Schedule / Calendar | 7 | `CRS_DEP_TIME`, `DISTANCE`, `DAY_OF_WEEK` | Core flight schedule known at booking |
| Raw Weather (NOAA) | 12 | `HourlyVisibility`, `HourlyWindSpeed`, `HourlyPrecipitation` | Point-in-time origin airport weather |
| Engineered Time | 8 | `hour_sin/cos`, `month_sin/cos`, `is_weekend` | Cyclical encoding preserves temporal proximity |
| Weather Composites | 5 | `weather_severity`, `has_precipitation`, `low_visibility` | Multi-factor adverse conditions individual variables miss |
| Congestion & Rolling | 4 | `origin_hourly_flights`, `tail_rolling_delay_rate` | Real-time airport load and aircraft cascade effects |
| Target Encoding | 4 | `carrier_te`, `origin_te`, `route_delay_rate` | Delay propensity per entity (train-only fit) |
| Multi-Timeframe RFM | 12 | `carrier_delay_rate_7d/14d/30d` | Short/medium/long-term disruption patterns |
| Graph-based | 5 | `origin_pagerank`, `total_degree`, `carrier_pagerank` | Network centrality and hub importance |
| Event-based | 3 | `is_holiday`, `is_national_event`, `holiday_proximity` | Calendar-driven demand surges |
| **Numeric subtotal** | **60** | | |
| Categorical (OHE) | \~753 | `ORIGIN`, `DEST`, `OP_UNIQUE_CARRIER`, `DEP_TIME_BLK` | Full categorical identity via one-hot encoding |
| **Total feature vector** | **\~813** | | Sparse (\~54 non-zeros per row) |

#### Hyperparameters and Settings Considered

**Stage 1: MLP Classification (PySpark MLlib)**

| Hyperparameter | Values Explored | Selected (MLP-C) |
| --- | --- | --- |
| Hidden layers | [64], [128], [128, 64] | [128, 64] |
| Solver | L-BFGS (only option in MLlib) | L-BFGS |
| Block size | 256 | 256 |
| Max iterations | 150 (with early stopping) | 150 (stopped at \~120) |
| Early stopping patience | 3 rounds x 10 iters | 3 |
| Class balancing | 1:1 random undersampling | 1:1 |
| Activation function | Sigmoid (MLlib default) | Sigmoid |

**Stage 1: GBT Classification (PySpark MLlib, Baseline)**

| Hyperparameter | Values Explored | Selected |
| --- | --- | --- |
| maxIter (trees) | 100 | 100 |
| maxDepth | 7 | 7 |
| stepSize (learning rate) | 0.1 | 0.1 |
| subsamplingRate | 0.8 | 0.8 |
| Classification threshold | 16 values: 0.20 to 0.65 (step 0.03) | 0.65 (F1-optimized) |
| Class balancing | Weighted (`weightCol` = class ratio 4.3:1) | Weighted |

**Stage 2: MLP Regression (PyTorch, single GPU)**

| Hyperparameter | Values Explored | Selected (MLP-B) |
| --- | --- | --- |
| Architecture | [813 to 64 to 1], [813 to 128 to 64 to 1] | [813 to 128 to 64 to 1] |
| Optimizer | Adam | Adam |
| Learning rate | 1e-3 with ReduceLROnPlateau | 1e-3 |
| Batch size | 4,096 | 4,096 |
| Epochs | Max 100, early stopping patience 5 | \~40 |
| Weight decay (L2) | 1e-5 | 1e-5 |
| Dropout | 0.1 | 0.1 |
| BatchNorm | Yes (after each linear layer) | Yes |
| Target transformation | Clip@P99 (359 min) | Clip@P99 |
| Activation | ReLU | ReLU |

**Stage 2: GBT Regression (PySpark MLlib)**

| Hyperparameter | Values Explored | Selected |
| --- | --- | --- |
| maxIter (trees) | 50, 100, 150, 200, 250 | 100 |
| maxDepth | 5, 6, 7, 8 | 5 |
| stepSize | 0.02, 0.03, 0.05, 0.1 | 0.05 |
| subsamplingRate | 0.7, 0.8 | 0.8 |
| Target variant | Clip@P99, No Clip, Log(delay+1) | Log(delay+1) |

#### Loss Functions

**Stage 1: MLP Classification** uses softmax cross-entropy (log loss):

$$\mathcal{L}_{\text{CE}} = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log(\hat{p}_i) + (1 - y_i) \log(1 - \hat{p}_i) \right]$$

where \\(y_i \in \{0, 1\}\\) is the true label (`DEP_DEL15`) and \\(\hat{p}_i\\) is the model's predicted probability of delay.

**Stage 1: GBT Classification** uses the logistic loss (deviance):

$$\mathcal{L}_{\text{GBT}} = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log(\sigma(f_i)) + (1 - y_i) \log(1 - \sigma(f_i)) \right]$$

where \\(f_i\\) is the ensemble's raw score for sample \\(i\\) and \\(\sigma(f_i) = 1 / (1 + e^{-f_i})\\) is the sigmoid function. Each successive tree is fitted to the negative gradient of this loss, performing gradient descent in function space.

**Stage 2: MLP Regression** uses mean squared error with L2 regularization:

$$\mathcal{L}_{\text{MSE+L2}} = \frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2 + \lambda \sum_{j} w_j^2$$

where \\(y_i\\) is the actual delay in minutes (clipped at P99 = 359 min), \\(\hat{y}_i\\) is the predicted delay, and \\(\lambda = 10^{-5}\\) is the weight decay coefficient. The L2 term penalizes large weights to prevent overfitting.

**Stage 2: GBT Regression (Log variant)** uses squared error on the log-transformed target:

$$\mathcal{L}_{\text{GBT-Log}} = \frac{1}{N} \sum_{i=1}^{N} \left( \log(y_i + 1) - \hat{f}_i \right)^2$$

where \\(\hat{f}_i\\) is the ensemble prediction in log-space. At inference, predictions are transformed back: \\(\hat{y}_i = e^{\hat{f}_i} - 1\\). The log transform compresses the heavy right tail of the delay distribution, helping the model allocate tree splits more evenly across the delay range.

#### Number of Experiments Conducted

| Pipeline | Stage | Experiments | Description |
| --- | --- | ---: | --- |
| MLP | Stage 1 Classification | 4 | MLP-A, MLP-B, MLP-C architectures (3-fold CV each) + MLP-D (No OHE comparison experiment) |
| MLP | Stage 2 Regression | 2 | MLP-A Shallow, MLP-B Deeper (PyTorch CPU) |
| GBT | Stage 1 Classification | 1 + 16 | 1 GBT config (tuned from prior runs) + 16-threshold grid search |
| GBT | Stage 2 Regression | 25 | 8 configs x 3 label variants + 1 extended check |
| **Total** | | **48** | Across both model families and both pipeline stages |


#### Experiments

All experiments were conducted on the same cluster configuration:

| Resource | Specification |
| --- | --- |
| Cloud Provider | AWS |
| Worker Node Type | m6g.xlarge (4 vCPUs, 16 GB RAM) |
| Driver Node Type | m6g.4xlarge (16 vCPUs, 64 GB RAM) |
| Number of Workers | 7 |
| Total Cluster Memory | 176 GB (7 x 16 GB + 64 GB driver) |
| GPU | 1 x NVIDIA T4 (16 GB VRAM) on driver only, used for PyTorch MLP regression |
| Spark Default Parallelism | 28 |
| Databricks Runtime | 17.3 LTS |

##### Stage 1: Classification Experiments (2019 Blind Test, 7,263,966 flights)

| # | Model | Architecture / Config | Pipeline | AUC-ROC | AUC-PR | F1 | Precision | Recall | Wall Time |
| ---: | --- | --- | --- | ---: | ---: | ---: | ---: | ---: | --- |
| 1 | MLP-A (Shallow) | [813 to 64 to 2], L-BFGS | MLP | 0.7854 | 0.4995 | 0.7665 | 0.8168 | 0.7435 | \~2.5 h |
| 2 | MLP-B (Wider) | [813 to 128 to 2], L-BFGS | MLP | 0.7859 | 0.5054 | 0.7868 | 0.8183 | 0.7701 | 2.3 h |
| 3 | **MLP-C (Deeper)** | **[813 to 128 to 64 to 2], L-BFGS** | **MLP** | **0.7845** | **0.5084** | **0.7834** | **0.8175** | **0.7657** | **2.7 h** |
| 4 | MLP-D (No OHE) | [60 to 128 to 64 to 2], L-BFGS | MLP | 0.7842 | 0.4915 | 0.7844 | 0.8184 | 0.7668 | 1.0 h |
| 5 | **GBT (Tuned)** | **100 trees, depth 7, lr=0.1** | **GBT** | **0.8045** | **0.5570** | **0.7912** | **0.8254** | **0.7740** | **2.5 h** |
| n/a | GBT + Threshold=0.65 | Same model, adjusted decision boundary | GBT | n/a | n/a | 0.5477 | 0.5292 | 0.5675 | instant |
| n/a | *Baseline (reference)* | *Phase 2 GBT, 12M data* | *Baseline* | *0.7970* | *0.5265* | *0.8206* | *0.8405* | *0.8082* | *n/a* |

**Best Stage 1 classifier: GBT (AUC-PR = 0.5570)** at default threshold 0.5.

##### Stage 2: Regression Experiments (Connected Pipeline, 2019 Blind Test)

| # | Model | Architecture / Config | Target Transform | MAPE | RMSE | MAE | Wall Time |
| ---: | --- | --- | --- | ---: | ---: | ---: | --- |
| 1 | MLP-A (Shallow) | [813 to 64 to 1], PyTorch Adam | Clip@P99 (359 min) | 101.50% | 63.23 | 44.71 | \~45 min |
| 2 | **MLP-B (Deeper)** | **[813 to 128 to 64 to 1], PyTorch Adam** | **Clip@P99 (359 min)** | **61.62%** | **74.00** | **46.36** | **\~50 min** |
| 3 | GBT-Reg (Clip@P99) | 100 trees, depth 5, lr=0.05 | Clip@P99 (359 min) | 90.50% | 73.36 | 48.54 | \~30 min ea. |
| 4 | GBT-Reg (No Clip) | Best of 8 configs | Raw target | 95.00% | 86.22 | 52.13 | \~30 min ea. |
| 5 | **GBT-Reg (Log)** | **100 trees, depth 5, lr=0.05** | **Log(delay+1)** | **65.85%** | **89.53** | **44.43** | **\~30 min ea.** |
| 6 | GBT-Reg (Log, ext.) | 150 trees, depth 6, lr=0.05 | Log(delay+1) | 65.85% | 89.53 | 44.43 | 87 min |
| n/a | *Baseline (reference)* | *Phase 2 GBT regressor* | *n/a* | *92.59%* | *74.30* | *41.28* | *n/a* |

**Best Stage 2 regressor: MLP-B Deeper (MAPE = 61.62%)**, selected because the primary metric is MAPE, which measures proportional accuracy on delay predictions rather than absolute error.

##### Overall Wall Time Summary

| Pipeline | Stage 1 | Stage 2 | OOF + Setup | Total |
| --- | --- | --- | --- | --- |
| MLP Pipeline | \~5.1 h (4 models x 3-fold CV + final retrain) | \~1.6 h (2 models) | \~0.5 h | \~7.2 h |
| GBT Pipeline | \~2.5 h (tuning + retrain + threshold search) | \~5.5 h (25 configs across 3 variants) | \~0.8 h (4-fold OOF) | \~8.8 h |
| **Grand total** | | | | **\~16 h** |


#### Modeling Decisions and Design Rationale

##### Encoding and Feature Representation

Before any model can train, every input must be numeric. Our pipeline one-hot encodes four categorical columns (origin airport, destination airport, carrier, and departure time block), expanding 4 text features into approximately 753 binary indicators. Combined with the 60 engineered numeric features, the final feature vector is approximately 813 dimensions, but only about 54 values are non-zero for any given flight. Spark stores this efficiently as a sparse vector, using roughly 15 times less memory than a dense representation.

This encoding choice raises a practical tradeoff. The 753 one-hot encoded columns represent 93% of the feature vector but carry less predictive signal per column than the 60 dense numeric features. The MLP-D comparison experiment (discussed in Results) quantified this directly: removing all 753 one-hot encoded columns only reduced AUC-PR by 0.017 (from 0.508 to 0.492), confirming that most delay-predictive information lives in the numeric features. Alternative encoding methods like feature hashing (mapping categories to fixed-size buckets, for example 128 instead of 753) or entity embeddings (learning dense low-dimensional representations that capture latent similarity between airports) could improve MLP performance and reduce memory pressure. We retained one-hot encoding because Spark ML's pipeline API natively supports it, sparse vector storage keeps memory manageable, and GBT, our best Stage 1 model, handles sparse inputs natively.

##### Scaling

We used StandardScaler with `withMean=False` for normalization. StandardScaler divides each feature by its standard deviation, which works well for roughly Gaussian distributions but compresses the informative range of skewed features. Precipitation is heavily zero-inflated (approximately 70% of readings are zero), wind gust speed has a long right tail, and rolling delay rates are right-skewed. For these features, the scaler overweights the common near-zero values at the expense of the informative non-zero range. Future improvements should include MinMaxScaler for bounded weather features (which have known physical ranges) and log transformations for heavy-tailed features. The `withMean=False` setting was necessary to preserve sparse vector storage, because subtracting the mean would fill every zero entry with a non-zero value, destroying sparsity and causing out-of-memory errors on our 16 GB workers.

Importantly, GBT is invariant to feature scale because tree splits depend on rank order, not magnitude. This means the scaling limitations described above do not affect GBT performance at all. MLP performance, by contrast, is sensitive to the quality of normalization, which is one factor in the classification gap between the two model families.

##### Training Strategy: Expanding-Window Cross-Validation

Within the training data (2015 to 2018), we use expanding-window cross-validation that accumulates all prior years into each successive fold:

| Fold | Training Years | Validation Year | Training Size |
| --- | --- | --- | --- |
| Fold 1 | 2015 | 2016 | \~5.7M |
| Fold 2 | 2015 to 2016 | 2017 | \~11.3M |
| Fold 3 | 2015 to 2017 | 2018 | \~16.8M |

The concern with expanding windows is staleness: if airline scheduling practices evolve over time, 2015 data may be outdated for predicting 2018 delays. A rolling window (for example, always using only the most recent 2 years) would handle this better by discarding old data, but at the cost of smaller training sets. We chose expanding windows because the delay rate is stable across years (17.2% to 18.7%), suggesting limited distributional drift. More training data also consistently improved MLP convergence, with Fold 3 at 16.8M rows outperforming Fold 1 at 5.7M. For a production deployment spanning many more years, a rolling window with a 2 to 3 year lookback would be more appropriate.

##### Classification: Why GBT Outperforms MLP

With the encoding, scaling, and training strategy established, we now turn to the classification results. GBT achieves AUC-PR = 0.557 versus the best MLP's 0.508, a consistent advantage across all classification metrics. This aligns with a well-established finding in machine learning: gradient-boosted trees tend to outperform neural networks on structured tabular data, particularly when the feature space mixes dense numeric columns with high-cardinality sparse categoricals.

GBT discovers feature interactions (for example, "Delta flights from ATL during peak hours in winter") through tree splits without requiring the model to learn them from weight combinations. MLPs must approximate these same interactions through layers of matrix multiplications on an 813-dimension vector that is 93% sparse zeros, which is inherently inefficient. GBT also ignores irrelevant features by simply not splitting on them, while MLPs must assign weights to all 813 inputs in the first hidden layer, and the vast majority of those weights receive near-zero gradient signal because their input is almost always zero.

The MLP-D comparison experiment confirms this interpretation. When the 753 sparse one-hot encoded columns are removed entirely, the MLP achieves comparable F1 (0.784 vs. 0.783) with 7 times fewer parameters. The sparse categorical dimensions were not helping the MLP learn; they were hindering it. The dense numeric features, including our graph-based PageRank and target encoding features, already capture the airport and carrier identity signals that one-hot encoding provides.

##### Regression: Why MLP Outperforms GBT

The roles reverse for Stage 2 regression. MLP-B achieves a MAPE of 61.62% compared to GBT's best of 65.85%. This reversal makes sense when we consider how each model type handles continuous output prediction and how the target distribution shapes model behavior.

Flight delay durations are heavily right-skewed: most delays cluster between 15 and 60 minutes (median 41 minutes), but extreme outliers stretch to 2,755 minutes (nearly two days). This skewed distribution creates a fundamental challenge for regression, and how we handle the long tail directly affects model quality. We compared three target transformation strategies to address this:

The first approach, Clip at the 99th percentile, caps extreme delays at around 300 to 359 minutes. The reasoning is that delays beyond 5 hours are operationally treated more like cancellations, so predicting the exact minute at that point adds little value. Clipping removes only the top 1% of delays while preserving the full shape of the remaining 99%. With this approach, GBT achieves a MAPE of 90.50%. The extreme tail no longer inflates the loss, but the target distribution is still heavily skewed toward shorter delays, so most of GBT's tree splits concentrate in the 15 to 60 minute range and underestimate longer delays.

The second approach, no clipping, retains the full distribution including outliers up to 2,755 minutes. This tests whether models can handle the raw data without any intervention. GBT's MAPE worsens to 95.00% under this approach, because the extreme outliers dominate the squared-error loss and pull tree splits toward accommodating rare very long delays at the expense of accurately predicting the common shorter ones.

The third approach, log transformation, trains GBT on log(delay + 1) instead of raw minutes. Taking the logarithm compresses the long right tail, making the delay distribution more symmetric and giving GBT a more evenly spread target to split on. Instead of allocating most tree splits to the dense 15 to 60 minute region, the log-scale model distributes splits more evenly across the full range. Predictions are converted back to minutes via exp(prediction) - 1 for evaluation. This is the most impactful change for GBT: MAPE drops from 90.50% (clipped) to 65.85% (log), a 25 percentage point improvement from a target transformation alone.

Despite the log transform substantially improving GBT, MLP-B with simple P99 clipping still achieves the best MAPE at 61.62%. The reason is architectural. GBT regression averages the target values of training examples that land in each leaf node, so it can only output a discrete set of values. For delay duration, where the target ranges continuously from 15 to over 300 minutes, this step-function behavior limits precision. MLPs, by contrast, output a single continuous number through a linear output layer. The network learns smooth, nonlinear mappings from features to delay duration, interpolating between training examples rather than binning them into leaves. The deeper architecture (813 to 128 to 64 to 1) learns hierarchical patterns: the first layer combines raw features into delay-relevant representations, the second layer distills those into a continuous minute estimate. Combined with the Adam optimizer and early stopping on validation error, the MLP converges to a smoother prediction surface that better captures the continuous nature of delay duration.


---

## Leakage Discussion

### What Is Data Leakage?

**Data leakage** occurs when information from outside the training dataset — typically from the future, the test set, or the target variable itself — is inadvertently used during model training. This creates an artificially optimistic view of model performance that collapses when deployed on genuinely unseen data.

Leakage falls into two main categories:

1. **Target leakage**: A feature directly encodes or is a proxy for the target variable. For example, using `ARR_DELAY` (arrival delay) or `CARRIER_DELAY` (carrier-attributed delay minutes) to predict departure delay would encode the outcome itself, since these columns are only available *after* the flight has departed and arrived.
2. **Temporal leakage**: Future information bleeds into the training process. For example, computing a rolling average delay rate that includes flights departing *after* the flight being predicted would give the model access to future outcomes.

### Hypothetical Leakage Example

Suppose we engineer a feature `avg_carrier_delay_same_day` that computes the average departure delay for a given airline **on the same day as the flight being predicted**. For a 6:00 AM flight, this feature would include delay data from 8:00 PM flights that haven’t departed yet. At training time, the model sees the full day’s data in one batch, so this feature appears highly predictive — it implicitly reveals same-day disruption patterns. At inference time on live data, those same-day future flights have not yet departed, making the feature unavailable or misleading. The model trained on this "good" feature would perform far worse in production than on the test set.

### Identified Sources of Mild Leakage

Our pipeline contains two features that introduce mild leakage. We acknowledge each, explain why the design decision was made, and propose the production-grade fix.

#### 1. Graph-Based PageRank Features (Temporal Leakage)

**What happened:** We compute PageRank scores on a directed flight network built from **all training years (2015–2018)** at once. The graph treats the four-year period as a single snapshot — edge weights (route frequencies) and node connectivity reflect the aggregate network structure across all training years. These static scores are then joined onto every flight in both training and test sets.

**Why this is leaky:** Consider a flight in the 2016 CV validation fold. Its `origin_pagerank` feature was computed from a graph that includes 2017 and 2018 flight routes — data from the *future* relative to 2016. In a strict temporal sense, a 2016 prediction should only use graph structure from 2015 and earlier. Similarly, any new route added after 2016 (e.g., a regional airport gaining a hub connection in 2018) would retroactively inflate that airport’s degree and PageRank for all years.

**Why we did it this way:** Computing separate per-year graphs would require maintaining 4 different PageRank lookup tables and adding a year-conditional join — increasing pipeline complexity and memory pressure on our constrained cluster. Since the US flight network structure is relatively stable year-over-year (no major airports opened or closed during 2015–2019, and the top-20 hubs remained largely unchanged), the temporal contamination is small. The graph captures *structural* network properties ("ATL is a major hub"), not delay-specific signals, so the information leaking is primarily topological rather than predictive.

**Proposed fix for production:** Compute PageRank in a **rolling or lagged manner** — for each year, build the graph from only preceding years (e.g., 2016 flights use a graph built from 2015 data only; 2017 uses 2015–2016). This ensures strict temporal consistency. Alternatively, recompute the graph monthly using a trailing 12-month window, which would also capture seasonal route changes.

#### 2. Scaling and Encoding Pipeline (Preprocessing Leakage)

**What happened**:
The Spark ML preprocessing pipeline (StringIndexer → OneHotEncoder → VectorAssembler → StandardScaler) was fitted on the full dataset before applying any temporal split. This means all encoding steps — including category indexing, one-hot vector construction, and feature scaling — were learned using data spanning the entire time range (2015–2019). Afterward, the dataset was split into training (2015–2018) and test (2019), and the pre-fitted pipeline was applied to both partitions.

**Why this is leaky**:
Fitting preprocessing transformations before a time-based split introduces temporal leakage because the encoders are informed by future data. For example, the StringIndexer assigns category indices based on global frequency, which includes 2019 observations that would not have been available at training time. Similarly, the StandardScaler computes means and variances using the full dataset, embedding information about the distribution of 2019 features into the transformation applied to earlier years. Even though these steps do not directly use the target variable, they allow subtle information from the future to influence feature representation, violating the principle that all preprocessing must be based solely on past data.

**Why we did it this way**:
This approach was initially taken for convenience and pipeline simplicity, as fitting a single preprocessing pipeline on the full dataset ensured consistent feature mappings and avoided repeated computation. At the time, the focus was on building a stable feature engineering workflow rather than strictly enforcing temporal separation during preprocessing.

**Impact assessment**:
The magnitude of leakage is relatively small. Category mappings from StringIndexer do not encode label information, and distributional shifts introduced by including future data in scaling are minor given the multi-year dataset. However, this still results in an optimistic bias, as the model benefits from slightly more informed feature transformations than would be available in a true production setting.

**Proposed fix for production**:
Refit the entire preprocessing pipeline strictly on the training period (2015–2018) after performing the temporal split. Then apply the fitted pipeline to both training and test datasets. This ensures that all feature transformations reflect only historically available information and preserves the integrity of time-based validation.



### Pipeline Leakage Audit (Clean Components)

The following pipeline components were verified to be **free of leakage**:

| Component | Verification |
| --- | --- |
| **Post-departure columns** | `ARR_DELAY`, `TAXI_OUT`, `CARRIER_DELAY`, `WEATHER_DELAY`, `NAS_DELAY`, `SECURITY_DELAY`, `LATE_AIRCRAFT_DELAY` — all dropped before feature engineering. Only pre-departure information retained. |
| **Temporal train/test split** | Year-based: 2015–2018 train, 2019 test. No random shuffling. 2019 data never appears in any model training fold. |
| **Cross-validation** | Expanding-window temporal CV (2015→2016, 2015–16→2017, 2015–17→2018). Validation year is always strictly after all training years. |
| **Rolling delay features** | All rolling windows (7d/14d/30d) use a strict **2-hour cutoff** before the scheduled departure time. Same-day future flights are excluded. |
| **Target encoding** | Mean delay rates per carrier/airport/route computed on **training data only**, applied as a static lookup to test data. |
| **Event features** | Derived from **static public calendars** (OPM federal holidays, NFL records), not from training data. |
| **Weather imputation** | Forward-fill and airport-month means computed within the training set only. |

### Cardinal Sins of Machine Learning — Compliance Check

| Cardinal Sin | Status | Explanation |
| --- | --- | --- |
| **Training on test data** | Not violated | All model training uses 2015–2018 exclusively. The 2019 blind test is evaluated exactly once per model. |
| **Using future data to predict the past** | Mild | Graph features use an aggregate 2015–2018 graph rather than per-year lagged graphs. This introduces mild temporal leakage for earlier CV folds. Impact is limited because graph features capture network structure (stable across years), not delay-specific signals. |
| **Leaking the target into features** | Not violated | Post-departure columns dropped. Target encoding uses train-set delay rates only. |
| **Fitting preprocessors before splitting** | Minor | StandardScaler and StringIndexer fitted on Phase 2 baseline split (includes 2019 Q1–Q3). Affects scaling parameters, not model learning. |
| **Overfitting to validation data** | Not violated | Final model selected on CV mean performance, not a single fold. 2019 blind test never used for model selection. |
| **Cherry-picking results** | Not violated | All experiment results reported, including underperforming variants. Best model selected by pre-defined primary metric. |

### Summary

Some features introduce mild leakage, particularly graph-based PageRank features computed at the yearly level and scaling applied prior to strict train-test separation. These were implemented for computational efficiency and exploratory purposes. In a production system, graph features would be recomputed in a rolling or lagged manner to ensure strict temporal consistency, and the encoding pipeline would be re-fitted on the training partition only. Neither source of leakage involves target-variable information, and the estimated metric impact is < 0.5% AUC-PR.


---

## Results and Discussion

The goal of this section is to present an interpretation of the key results from all experiments conducted across the MLP and GBT pipelines, compare performance across stages and phases, and provide a broader analysis of what these results mean for flight delay prediction.

### Metrics Discussion

#### Why AUC-PR as the Primary Classification Metric

With a 19% positive rate (4.3:1 class imbalance), AUC-ROC is misleading. AUC-ROC's false positive rate denominator includes \~5.9M true negatives, so even millions of false positives barely move the curve. AUC-PR uses precision in the denominator, directly penalizing false alarms. A random classifier achieves AUC-PR = 0.19 (the base rate); our best model's 0.557 represents a 2.9x lift over random, which is operationally meaningful for proactive delay management.

We also report AUC-ROC for comparability with prior literature, and F1/Precision/Recall at the default threshold (0.5) for interpretability.

#### Why MAPE Over RMSE for Regression

Flight delays are heavily right-skewed: most delays are 15 to 60 minutes, but some exceed 500 minutes. Mean-based metrics (MAE, RMSE) are pulled upward by extreme outliers, painting a misleadingly pessimistic picture of typical performance. Consider:

* A 10-minute error on a 20-minute delay (50% MAPE) is operationally worse than a 10-minute error on a 200-minute delay (5% MAPE)
* RMSE squares errors, so a single 500-minute outlier contributes as much to the metric as hundreds of 15-minute errors
* MAE treats all absolute errors equally regardless of whether the delay is 15 minutes or 300 minutes

MAPE measures proportional accuracy, which aligns with how delays are experienced: the relative error matters more than the absolute error.

---

### Stage 1: Binary Classification Results

Stage 1 asks: "Will this flight depart 15 or more minutes late?" We evaluated five classifier variants on the 2019 blind test (7,263,966 flights, natural \~19% delay rate).

#### Cross-Model Comparison (2019 Blind Test)

| Model | Architecture | AUC-ROC | AUC-PR | F1 | Precision | Recall |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| MLP-A (Shallow) | [813 to 64 to 2] | 0.7854 | 0.4995 | 0.7665 | 0.8168 | 0.7435 |
| MLP-B (Wider) | [813 to 128 to 2] | 0.7859 | 0.5054 | 0.7868 | 0.8183 | 0.7701 |
| MLP-C (Deeper) | [813 to 128 to 64 to 2] | 0.7845 | 0.5084 | 0.7834 | 0.8175 | 0.7657 |
| MLP-D (No OHE) | [60 to 128 to 64 to 2] | 0.7842 | 0.4915 | 0.7844 | 0.8184 | 0.7668 |
| **GBT (Tuned)** | **100 trees, depth 7** | **0.8045** | **0.5570** | **0.7912** | **0.8254** | **0.7740** |

#### Training vs. Test Performance

To confirm that our models generalize well and are not simply memorizing the training data, we compare training, cross-validation, and blind test metrics side by side.

| Model | Train AUC-PR | CV Mean AUC-PR | 2019 Test AUC-PR |
| --- | ---: | ---: | ---: |
| MLP-C (Deeper) | 0.800 | 0.483 | 0.508 |
| MLP-D (No OHE) | 0.770 | 0.475 | 0.492 |
| GBT (Tuned) | n/a | 0.567 (val 2018) | 0.557 |

The GBT classifier shows tight alignment between its validation score and the blind test (0.567 vs. 0.557), confirming stable generalization with no signs of overfitting. The MLP classifiers show a larger gap between train and test AUC-PR, which is expected with L-BFGS full-batch optimization that naturally fits the training distribution more tightly. Importantly, the cross-validation means closely track the blind test scores (within 0.02), which validates our expanding-window temporal CV design as a reliable estimator of real-world performance. This means the model is not overfitting to the point of unreliable predictions; the CV process correctly estimates what the model will achieve on unseen future data.

#### Key Findings

**1. GBT outperforms all MLP variants on every metric.** The GBT classifier achieves AUC-PR = 0.5570 versus the best MLP's 0.5084, a +9.6% relative improvement. This is consistent with the broader ML literature showing that gradient-boosted trees tend to outperform neural networks on tabular data, especially when the feature space is a mix of dense numerics and sparse categoricals. GBT's ability to natively handle feature interactions through tree splits gives it an advantage over the MLP's weight-based learning.

**2. MLP architectures plateau at \~0.785 AUC-ROC regardless of width or depth.** Going from 64 to 128 neurons (MLP-A to MLP-B) added only +0.006 AUC-PR. Adding a second hidden layer (MLP-B to MLP-C) added only +0.003 more. This indicates a signal ceiling: the predictive information available 2 hours before departure imposes a hard limit on discriminative performance that additional model capacity cannot overcome.

**3. All models achieve high precision (\~0.82) with moderate recall (\~0.74 to 0.77).** This precision-first behavior is intentional for our pipeline: Stage 2 regression only runs on predicted-delayed flights, so high precision ensures Stage 2 receives a clean subset with minimal false positives contaminating the regression training.

**4. Temporal CV closely tracks blind-test performance.** MLP CV AUC-PR means (0.48 to 0.50) align within 0.02 of the 2019 test scores (0.49 to 0.51), validating the expanding-window design as a reliable estimator. No model showed a surprising test-time drop, confirming that the 2019 delay distribution is representative of 2015 to 2018 patterns.

#### MLP-D: Classification Without Categorical Encoding

To understand whether the 753 one-hot encoded categorical columns are actually helping the neural network, we trained MLP-D using the same deeper architecture as MLP-C ([128, 64] hidden layers) but removed all one-hot encoded columns entirely. Instead of the full 813-dimensional sparse feature vector, MLP-D receives only the 60 dense numeric features (schedule, weather, engineered time, composites, congestion, target encoding, RFM, graph, and event features).

The results show that MLP-D achieves F1 = 0.784 and precision = 0.818, essentially matching MLP-C (F1 = 0.783, precision = 0.818) despite having 7 times fewer parameters (approximately 16K vs. 112K) and training 3.5 times faster (59 minutes vs. 161 minutes). The AUC-PR gap between MLP-D and MLP-C is only 0.017 (0.492 vs. 0.508).

This finding has two important takeaways. First, the dense numeric features, particularly the graph-based PageRank scores and target-encoded delay rates, already capture much of the airport and carrier identity signal that the 753 one-hot columns encode. The PageRank features summarize how important and well-connected each airport is, while the target encoding features capture each entity's historical delay tendency, effectively representing what the one-hot columns represent in a far more compact form. Second, the sparse one-hot dimensions may actually make learning harder for the MLP. The network must assign weights to all 813 inputs in the first hidden layer, and the vast majority of those weights receive near-zero gradient signal because their inputs are almost always zero. This dilutes the learning signal and contributes to the train/test gap observed across all MLP variants.

For future work, this result motivates exploring alternative categorical encoding strategies such as entity embeddings or feature hashing, which could provide the categorical identity signal in a dense, low-dimensional form that is more compatible with neural network optimization.

#### Threshold Optimization (GBT)

The GBT pipeline includes a 16-point threshold grid search (0.20 to 0.65). At the default threshold 0.5, F1 = 0.791. Raising the threshold to 0.65 increases precision to 0.529 but reduces recall to 0.568 (F1 = 0.548). This trade-off is operationally configurable: a cost-sensitive airline might prefer the high-precision operating point (fewer false alarms, less wasted rebooking), while a passenger-centric airline might lower the threshold for broader delay coverage.


### Stage 2: Delay Duration Regression Results

Stage 2 asks: "How many minutes late will this flight be?" Only flights that Stage 1 classifies as delayed are passed to Stage 2. We evaluated two model families across multiple target transformations.

#### Cross-Model Comparison (Connected Pipeline, 2019 Blind Test)

| Model | Target Transform | MAPE | RMSE | MAE |
| --- | --- | ---: | ---: | ---: |
| MLP-A (Shallow) | Clip@P99 | 101.50% | 63.23 | 44.71 |
| **MLP-B (Deeper)** | **Clip@P99** | **61.62%** | **74.00** | **46.36** |
| GBT-Reg (Clip@P99) | Clip@P99 | 90.50% | 73.36 | 48.54 |
| GBT-Reg (No Clip) | Raw target | 95.00% | 86.22 | 52.13 |
| GBT-Reg (Log) | Log(delay+1) | 65.85% | 89.53 | 44.43 |
| *Baseline (Phase 2)* | *n/a* | *92.59%* | *74.30* | *41.28* |

#### Training vs. Test Performance

| Model | Train MAPE | Validation MAPE | 2019 Test MAPE |
| --- | ---: | ---: | ---: |
| MLP-B (Deeper) | 60.47% | 63.51% | 61.62% |
| GBT-Reg (Log) | n/a | n/a | 65.85% |

The MLP-B regressor shows minimal gap between training, validation, and test MAPE (60.5% vs. 63.5% vs. 61.6%), indicating strong generalization with no overfitting. The test MAPE is actually slightly better than the validation MAPE, which further confirms the model is not memorizing training patterns. Early stopping (patience=3 on validation RMSE) effectively prevents overtraining.

#### Key Findings

**1. MLP-B Deeper achieves the best MAPE (61.62%).** This is the stronget performance in the entire pipeline and demonstrates that continuous neural network regression (MSE loss, Adam optimizer) substantially outperforms tree-based regression for this right-skewed target distribution. The smooth nonlinear mapping learned by the MLP better handles the transition from small delays (15 to 30 min) to moderate delays (60 to 120 min).

**2. Log-transformed GBT (MAPE = 65.85%) is a strong runner-up.** Training GBT on `log(delay + 1)` and exponentiating predictions back to minutes compresses the heavy right tail, letting the tree allocate splits more evenly. Without this transform, GBT Clip@P99 and No Clip variants perform significantly worse (90.5% and 95.0% MAPE), confirming that target transformation is critical for tree-based regressors on skewed distributions.

**3. The RMSE vs. MAPE trade-off reveals model behavior differences.** MLP-B has the best MAPE (61.6%) but a higher RMSE (74.0) than MLP-A (63.2 RMSE, 101.5% MAPE). This means MLP-B makes smaller proportional errors on the common 15 to 60 minute delays (where MAPE matters) but occasionally makes larger absolute errors on extreme delays (which inflate RMSE). For operational use, proportional accuracy (MAPE) is more actionable because a 10-minute error on a 20-minute delay is worse than a 10-minute error on a 200-minute delay.

**4. Target clipping at the 99th percentile (359 min) helps MLP but is insufficient for GBT.** MLP-B benefits from Clip@P99 because MSE loss is sensitive to outliers, and capping extreme values prevents gradient explosions. GBT, which already handles outliers through tree partitioning, needs the stronger Log(delay+1) transform to achieve comparable MAPE.

---

### Phase-Over-Phase Comparison

| Metric | Phase 2 (12M, 1 year) | Phase 3 Best | Improvement |
| --- | --- | --- | --- |
| Training data | 5.8M flights | 23.9M flights | 4.1x more data |
| Feature vector | 727 dim, 52 numeric | 813 dim, 60 numeric | +8 graph/event features |
| Stage 1 AUC-PR | 0.5265 (GBT) | 0.5570 (GBT) | +5.8% relative |
| Stage 1 F1 | 0.8206 | 0.7912 | -3.6% (different threshold regime) |
| Stage 2 MAPE | 92.59% | 61.62% (MLP-B) | -31.0 percentage points |
| Stage 2 MAE | 41.28 min | 44.43 min (GBT-Log) | +3.2 min (different metric focus) |

The pipeline trades a small F1 decrease for a substantially higher AUC-PR and dramatically better MAPE. The F1 difference is partly explained by the different class balancing strategies (using weightCol versus 1:1 undersampling) and the change from quarter-based to year-based splits.

---

### Overall Pipeline Discussion

The two-stage design is validated by the results. Stage 1 identifies delayed flights with 82.5% precision (at default threshold), ensuring that Stage 2 receives a clean subset of genuinely delayed flights. Stage 2 then predicts delay duration with 61.6% MAPE, meaning for a 60-minute delay, the model predicts within roughly plus or minus 37 minutes. While not precise enough for gate-level scheduling, this is sufficient for proactive passenger rebooking (any advance warning is better than none), crew repositioning (buffer allocation can absorb the prediction uncertainty), and operational dashboard alerting (flag flights likely to exceed 1-hour delay).

Architecture scaling offers diminishing returns, but feature engineering does not. The four MLP architectures have different parameter count with the features included and size, but only have a 0.017 range in AUC-PR. In contrast, adding 8 graph and event features (Phase 2 to Phase 3) improved AUC-PR by 0.031 on the same GBT model. This suggests future gains are more likely from richer data sources (real-time ASPM airport throughput, gate-level congestion, crew scheduling data) than from deeper or wider networks.

The signal ceiling for pre-departure delay prediction is approximately AUC-ROC of 0.80. Both MLP and GBT converge near this value despite fundamentally different learning approaches (gradient descent vs. greedy tree splitting). This ceiling is inherent to the task: many delays are caused by events that occur after the 2-hour prediction window (runway incidents, late-arriving inbound aircraft, last-minute weather changes). No model architecture can predict information that does not yet exist in the features.

**Model selection depends on the deployment context:**

| Criterion | Best Choice | Rationale |
| --- | --- | --- |
| Stage 1 accuracy | GBT | Highest AUC-PR (0.557), native feature interactions |
| Stage 2 accuracy | MLP-B (PyTorch) | Lowest MAPE (61.6%), smooth regression surface |
| Training speed | GBT end-to-end | Single distributed framework, no GPU required |
| Inference speed | GBT | Tree traversal is faster than matrix multiplication |
| Resource efficiency | MLP-D (No OHE) | 7x fewer params, comparable F1, no OHE overhead |
| Threshold flexibility | GBT | Post-hoc threshold tuning without retraining |


### Limitations

1. **Regression difficulty is inherent.** Flight delay duration depends on cascading factors that emerge after the 2-hour prediction window (runway incidents, late inbound aircraft, ATC ground stops). No feature set available at prediction time can fully capture these. The signal ceiling is real: both GBT and MLP converge at approximately 0.80 AUC-ROC and 62 to 66% MAPE despite fundamentally different architectures.

2. **Missing data sources.** Our feature set lacks critical operational variables that airlines have internally: crew scheduling and duty-hour limits, real-time gate assignments and turnaround times, inbound aircraft tracking (late-arriving planes are the number one delay cause), and ASPM real-time airport throughput. Adding these would likely improve performance more than any modeling change.

3. **Leakage in graph features and scaler.** As detailed in the Leakage Discussion, PageRank features were computed on the aggregate 2015 to 2018 graph rather than in a rolling or lagged manner, and the StandardScaler was fitted before strict train/test separation. Both introduce mild temporal contamination that does not affect target-variable information but deviates from production-grade holdout discipline.

4. **Mean-based metrics obscure typical performance.** Our primary regression metrics (MAPE, MAE, RMSE) are all mean-based and inflated by the heavy tail of extreme delays. Median metrics (MdAE, Median MAPE) would provide a more representative picture of how the model performs on the typical delayed flight.

5. **No uncertainty quantification.** The pipeline produces point predictions only. Operations managers need confidence intervals (for example, "this flight will be 45 plus or minus 20 minutes late with 90% confidence") to make risk-appropriate decisions. Conformal prediction or quantile regression would address this.


---

## Next Steps and Future Work

The most immediate priority is to fix the two identified sources of mild leakage. The StandardScaler and the full encoding pipeline (StringIndexer, OHE, VectorAssembler, StandardScaler) should be re-fit exclusively on the year-based training partition (2015 to 2018), rather than reusing the Phase 2 pre-fitted pipeline. This is a straightforward change that would take approximately 10 minutes of compute time. Additionally, the graph-based PageRank features should be computed in a rolling or lagged manner, where each year's predictions use a graph built from only the preceding years. For example, predictions for 2017 flights would use a graph built from 2015 to 2016 data only. This ensures strict temporal consistency and eliminates the mild leakage from our current aggregate graph approach.

On the feature engineering side, several improvements to scaling and transformation would benefit the pipeline. MinMaxScaler should be used for weather features that have natural physical bounds (visibility 0 to 10 miles, wind speed 0 to 100 mph, temperature negative 40 to 120 degrees Fahrenheit), since StandardScaler assumes Gaussian distributions that zero-inflated precipitation and heavy-tailed wind gusts violate. Log transformations of the form log(x + 1) should be applied to heavy-tailed features like precipitation and rolling delay rates before scaling. For tree-based models specifically, label encoding could replace the 753 OHE columns with just 4 integer-encoded features, since GBT can natively split on integer-encoded categories without needing binary expansion.

For the regression target in Stage 2, our experiments tested clipping and log transformation separately, but combining both could yield further gains. Applying clipping before log transformation would first cap extreme delays at P99 (359 minutes) to remove outliers, then apply log(delay + 1) to compress the remaining distribution. This combines outlier removal with distributional normalization and could potentially yield better MAPE than either technique alone.

Future evaluation should include median-based metrics (Median Absolute Error and Median MAPE) alongside the current mean-based metrics, since the heavy skew in the delay distribution means that median metrics better reflect typical model performance. Per-bucket accuracy across delay ranges (15 to 30 minutes, 30 to 60 minutes, 60 to 120 minutes, and 120+ minutes) would also reveal whether the model is accurate on the operationally actionable short delays even if aggregate MAPE is inflated by extreme outliers.

The one-hot encoding dimensionality issue, highlighted by the MLP-D experiment, motivates exploring alternative categorical encoding methods. Entity embeddings could learn dense 16 to 32 dimensional airport representations that capture latent similarity between airports, reducing the feature vector from 813 to approximately 130 dimensions. Feature hashing offers a fixed-size feature space (for example, 128 buckets) that handles unseen categories at inference time. Frequency encoding simply replaces each category with its count and carries no target leakage risk.

Reducing the prediction lookahead from two hours to one hour would provide more current weather and congestion signals, likely improving accuracy at the cost of less advance warning. A dual-window approach with an initial prediction at two hours and a refined prediction at one hour would give earlier warning followed by a more accurate update as departure approaches. The one-hour model could incorporate additional features like real-time gate utilization and taxi queue lengths.

Finally, several model architecture improvements could strengthen the pipeline. Ensemble stacking could combine GBT (best for Stage 1) and MLP-B (best for Stage 2) with a meta-learner for calibrated predictions. Online learning would allow incremental retraining as new data arrives rather than static batch retraining. Conformal prediction could add calibrated prediction intervals to Stage 2 for risk-aware decision-making, giving operations managers the confidence bounds they need rather than just point estimates.


## Conclusion

### Project Focus and Importance

Flight delays cost the U.S. aviation industry billions of dollars annually through cascading gate conflicts, crew scheduling disruptions, and passenger re-accommodation. Despite these costs, most airports rely on reactive delay management rather than predictive systems. Our project focuses on building a two-stage machine learning pipeline that predicts both the occurrence and the duration of flight departure delays using only pre-departure information available two hours before scheduled departure. This is important because even modest advance warning of delays enables operations teams to proactively reassign gates, reposition crews, and rebook passengers before disruptions cascade through the network.

---
### Hypothesis

Our hypothesis is that a machine learning pipeline with custom-engineered features, including temporal, weather, congestion, graph-based network centrality, and event-driven features, can accurately predict whether a flight will be delayed by 15 or more minutes and estimate the duration of that delay. We further hypothesized that combining a tree-based classifier (GBT) with a neural network regressor (MLP) in a connected two-stage architecture would outperform any single-model approach by leveraging each model family's strengths.

---
### Summary of Main Points

**Best Features:** Among the 60 engineered numeric features, the most impactful families were congestion and rolling delay features (particularly `tail_rolling_delay_rate`, which captures the late-aircraft cascade effect), multi-timeframe RFM features (7-day, 14-day, and 30-day rolling delay rates that detect short-term disruptions), and graph-based PageRank features that quantify airport importance in the network. The graph features revealed that major airports like ATL, ORD, DFW, and LAX carry the highest centrality scores and confirmed that carrier-specific network structure (centralized vs. distributed routing) is a meaningful predictor. Target-encoded delay rates per carrier, origin, destination, and route also contributed meaningful signal without inflating dimensionality.

**Best Models and Hyperparameters:** For Stage 1 classification, GBT served as our baseline and ultimately our best classifier, achieving AUC-PR = 0.557, F1 = 0.791, precision = 0.825, and recall = 0.774 on the 2019 blind test (7.26 million flights). The winning GBT configuration was 100 trees, max depth 7, learning rate 0.1, subsampling rate 0.8, with class weighting (4.3:1) to handle the imbalanced dataset. For Stage 2 regression, MLP-B Deeper (architecture: 813 to 128 to 64 to 1, trained with PyTorch using Adam optimizer, learning rate 1e-3, batch size 4,096, and Clip@P99 target transformation) achieved the best MAPE of 61.62%, representing a 31 percentage point improvement over the Phase 2 baseline of 92.59%.

**Pipeline Architecture.** The two-stage connected design was validated by the results. Stage 1 identifies delayed flights with 82.5% precision at the default threshold, ensuring Stage 2 receives a clean subset. Stage 2 then predicts delay duration with an MAE of approximately 46 minutes. The out-of-fold connected training approach ensured the regression model was trained on realistic, noisy classifier outputs rather than artificially clean in-sample predictions.

---
### Significance of Results

Our results demonstrate that pre-departure features alone provide a meaningful signal for delay prediction, despite the inherent unpredictability of events that occur after the prediction window. The Phase 3 pipeline improved Stage 1 AUC-PR by 5.8% relative to Phase 2 and reduced Stage 2 MAPE by 31 percentage points, showing that both more training data (23.9M vs. 5.8M flights) and richer feature engineering (graph and event features) contribute to better predictions.

The GBT vs. MLP comparison yielded an important finding: GBT consistently outperformed MLP for classification on this tabular dataset, consistent with the ML literature on structured data. However, MLP outperformed GBT for regression, achieving the lowest MAPE through its smooth nonlinear mapping of the right-skewed delay distribution. This suggests that the optimal model family depends on the task structure, and combining both in a connected pipeline is the strongest approach.

The MLP-D experiment (No OHE, 60-dim input) further showed that removing all 753 one-hot encoded columns only reduced AUC-PR by 0.017 while cutting parameters by 7x, confirming that most delay-predictive information is captured by the dense numeric features and that alternative encoding methods (embeddings, hashing) are a promising direction.

---
### Additional Contribution: Neural Network Regression

As an additional contribution beyond the core classification pipeline, we implemented PyTorch-based MLP regression models for Stage 2. Since Spark MLlib does not provide a native MLP regressor, we built a custom training loop using PyTorch with MSE loss, Adam optimization, batch normalization, dropout regularization, and early stopping. The training data was converted from Spark ML sparse vectors to scipy CSR matrices and processed on the driver node CPU. This neural network regression component produced our best Stage 2 results (MAPE = 61.62%) and demonstrates the value of extending deep learning approaches beyond classification to continuous delay duration prediction.

---
### Future of the Project
Several directions could further improve the pipeline. First, the identified leakage issues (PageRank computed at the yearly level and StandardScaler fitted before the year-based split) should be corrected by computing graph features in a rolling or lagged manner and re-fitting the encoding pipeline on training data only. Second, combining Clip@P99 with the log transform for Stage 2 regression could yield further gains, as each technique addresses different aspects of the skewed target distribution. Third, reducing the prediction lookahead from two hours to one hour would provide more current weather and congestion signals, likely improving accuracy at the cost of less advance warning. Fourth, integrating real-time data sources not available in the OTPW dataset, such as crew duty hours, inbound aircraft position, and gate-level congestion, would address the fundamental signal ceiling we observed (AUC-ROC approximately 0.80 across all model families). Finally, graph neural networks that model dynamic delay propagation across the airport network, rather than static yearly PageRank, represent the most promising long-term direction for capturing the cascading nature of flight delays.